# MOLM Component and Architecture Ablation

This notebook evaluates sharing, ranking/gap losses, routing, conditional shared-gradient projection, task-private paths, and capacity controls on the downstream fixed-budget multi-objective candidate-prioritization endpoint.

### Suite A — eight-arm architecture/component ablation

The architecture suite isolates combinations of shared learning, dominance-aware ordering, conditional gradient projection, task-private adapters, an independent loss-matched control, and shared-capacity controls.

### Suite B — ranking/gap loss ablation

The loss suite compares focal only, focal+ranking, focal+gap, and focal+ranking+gap.

### Pareto protocol

All arms are evaluated under identical candidate budgets `K={5,10,15,20,25}` using Recall, Precision, Enrichment, hypervolume, and IGD.

### Representation

The controlled suite uses the representation specified in the notebook configuration so that component effects are assessed within a common feature space.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, time, platform, queue, threading, zipfile
import numpy as np
import pandas as pd

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/molm_pareto_component_ablation')
REPO = WORK_ROOT / 'repo'
CODE_DIR = WORK_ROOT / 'code'
FEATURE_DIR = WORK_ROOT / 'features'
ARCH_RUN_DIR = WORK_ROOT / 'runs_architecture'
LOSS_RUN_DIR = WORK_ROOT / 'runs_loss'
ANALYSIS_DIR = WORK_ROOT / 'analysis'
LOG_DIR = WORK_ROOT / 'logs'
for p in [WORK_ROOT, CODE_DIR, FEATURE_DIR, ARCH_RUN_DIR, LOSS_RUN_DIR, ANALYSIS_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

REPO_URL = 'https://github.com/DigantaX/molm-pipeline.git'
PINNED_COMMIT = 'c5923984f0d5176977edb4a4ffd8fc5f98536043'
SEEDS = [42, 123, 456, 789, 2024]
FEATURES = ['onehot']
K_VALUES = [5, 10, 15, 20, 25]
EPOCHS = 25
BATCH_SIZE = 64
LEARNING_RATE = 5e-5

# Exact defaults used by the completed A-H component ablation.
ARCH_COMPONENT_SETTINGS = {
    'DOMINANCE_WEIGHT':0.03,
    'DOMINANCE_MARGIN':0.20,
    'DOMINANCE_WARMUP':5,
    'DOMINANCE_MAX_PAIRS':4096,
    'PRIVATE_ADAPTER_DIM':32,
    'PRIVATE_GATE_INIT':-2.0,
    'ROUTING_EPS':1e-12,
}

REQUIRE_TWO_GPUS = True
RESUME = True

ARCH_ARMS = [
    'shared_base',
    'shared_dom',
    'shared_dom_pcgrad',
    'shared_dom_private',
    'full_routed',
    'independent_st_dom',
    'shared_capacity_dom',
    'shared_capacity_dom_pcgrad',
]
ARCH_ARM_LABELS = {
    'shared_base':'A Shared Base MOLM',
    'shared_dom':'B Shared + Dominance',
    'shared_dom_pcgrad':'C Shared + Dominance + PCGrad',
    'shared_dom_private':'D Shared + Dominance + Private',
    'full_routed':'E Full Routed-MOLM',
    'independent_st_dom':'F Independent-parameter + matched joint loss',
    'shared_capacity_dom':'G Capacity-Matched Shared + Dominance',
    'shared_capacity_dom_pcgrad':'H Capacity-Matched Shared + Dominance + PCGrad',
}

LOSS_ARMS = [
    'S0_shared_focal',
    'SR_shared_focal_ranking',
    'SG_shared_focal_gap',
    'SRG_shared_full',
    'IRG_independent_full',
]
LOSS_ARM_LABELS = {
    'S0_shared_focal':'S0 Shared + Focal',
    'SR_shared_focal_ranking':'SR Shared + Focal + Ranking',
    'SG_shared_focal_gap':'SG Shared + Focal + Gap',
    'SRG_shared_full':'SRG Shared + Focal + Ranking + Gap',
    'IRG_independent_full':'IRG Independent + Focal + Ranking + Gap',
}
ALL_ARMS = ARCH_ARMS + LOSS_ARMS
ARM_LABELS = {**ARCH_ARM_LABELS, **LOSS_ARM_LABELS}

# Locked 42-sample subset used in the prior unified analysis.
IGG_PRIMARY42_SAMPLE_IDS = [
 '14.53','18.06','24.32','27.02','27.1','27.22','27.32','43.01','43.06','43.14','43.2',
 '45.02','45.04','EM01','EM02','EM04','EM13','R2I1','R2I2','R2I3','R2I4','R2I5','R2I6',
 'R2I7','R2I8','R2I9','R2I10','R2I11','R2I12','R2I13','R2I14','R2I15','R2I16','R2I17',
 'R2I18','R2I19','R2I20','R2I21','R2I22','R2I23','R2I24','WT'
]

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Work root:', WORK_ROOT)
print('Architecture arms:', len(ARCH_ARMS))
print('Loss arms:', len(LOSS_ARMS))
print('Total fits per feature across 5 seeds:', len(SEEDS)*(len(ARCH_ARMS)+len(LOSS_ARMS)))


## 1. Pinned repository and GPU preflight
The notebook uses the same pinned repository commit as the unified extended experiment suite. If a repository snapshot containing `phase0_config.py` and the data files is already mounted under `/kaggle/input`, it is copied; otherwise the notebook clones the public repository and checks out the pinned commit.

In [ ]:
import torch
if REQUIRE_TWO_GPUS and torch.cuda.device_count() != 2:
    raise RuntimeError(f'Select Kaggle T4 x2. Found {torch.cuda.device_count()} GPU(s).')
print('GPUs:', [torch.cuda.get_device_properties(i).name for i in range(torch.cuda.device_count())])

repo_candidates=[]
if INPUT_ROOT.exists():
    for p in INPUT_ROOT.rglob('phase0_config.py'):
        parent=p.parent
        if (parent/'data'/'emi_binding.csv').exists() and (parent/'data'/'iso_binding.csv').exists():
            repo_candidates.append(parent)

if repo_candidates:
    if REPO.exists(): shutil.rmtree(REPO)
    shutil.copytree(repo_candidates[0], REPO)
    print('Using repository snapshot from input:', repo_candidates[0])
else:
    if REPO.exists(): shutil.rmtree(REPO)
    subprocess.run(['git','clone',REPO_URL,str(REPO)],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout',PINNED_COMMIT],check=True)

if (REPO/'.git').exists():
    commit=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    if commit != PINNED_COMMIT:
        raise RuntimeError(f'Repository commit mismatch: {commit}')
print('Repository ready:', REPO)

## 2. Load EMI / ISO / IgG and build the controlled OneHot feature archive
No ESM computation is needed in this notebook. This is intentional: the purpose is to isolate **training objective and parameter-sharing effects**, not representation effects.

In [ ]:
sys.path.insert(0, str(REPO))
import phase0_config as repo_config

repo_cfg = getattr(repo_config, 'config', repo_config)
print('Pinned loss settings:')
for k in ['RANKING_WEIGHT_AFF','RANKING_WEIGHT_SPEC','GAP_WEIGHT_AFF','GAP_WEIGHT_SPEC','RANKING_MARGIN','GAP_MARGIN','FOCAL_GAMMA']:
    print(' ', k, getattr(repo_cfg, k))

EXPECTED_LOSS = {
    'RANKING_WEIGHT_AFF':0.3,
    'RANKING_WEIGHT_SPEC':0.6,
    'GAP_WEIGHT_AFF':0.2,
    'GAP_WEIGHT_SPEC':0.6,
    'RANKING_MARGIN':0.3,
    'GAP_MARGIN':0.2,
    'FOCAL_GAMMA':2.0,
}
for key, expected in EXPECTED_LOSS.items():
    observed=float(getattr(repo_cfg,key))
    if not np.isclose(observed, expected):
        raise RuntimeError(f'{key} changed: {observed} != {expected}')


def resolve_column(frame, candidates):
    for c in candidates:
        if c in frame.columns:
            return c
    raise KeyError((candidates, list(frame.columns)))

EMI_CSV=REPO/'data'/'emi_binding.csv'
ISO_CSV=REPO/'data'/'iso_binding.csv'
IGG_CSV=REPO/'data'/'igg_binding.csv'
emi_binding=pd.read_csv(EMI_CSV,index_col=0)
iso_binding=pd.read_csv(ISO_CSV,index_col=0)
igg_binding=pd.read_csv(IGG_CSV,index_col=0)

emi_sequences=emi_binding.index.astype(str).to_numpy()
iso_sequences=iso_binding.index.astype(str).to_numpy()
igg_sequences=igg_binding.index.astype(str).to_numpy()

emi_aff_col=resolve_column(emi_binding,['ANT Binding','ANT','Affinity','affinity'])
emi_ova_col=resolve_column(emi_binding,['OVA Binding','OVA','PSY','Specificity','specificity'])
iso_aff_col=resolve_column(iso_binding,['ANT Binding','ANT','Affinity','affinity'])
iso_ova_col=resolve_column(iso_binding,['OVA Binding','OVA','PSY','Specificity','specificity'])
igg_aff_col=resolve_column(igg_binding,['ANT Binding','ANT','Affinity','affinity'])
igg_ova_col=resolve_column(igg_binding,['OVA Binding','OVA','PSY','Specificity','specificity'])

y_aff_emi=(emi_binding[emi_aff_col].to_numpy()>0).astype(np.float32)
y_ova_emi=(emi_binding[emi_ova_col].to_numpy()>0).astype(np.float32)
iso_aff=iso_binding[iso_aff_col].to_numpy(float)
iso_ova=iso_binding[iso_ova_col].to_numpy(float)
igg_aff=igg_binding[igg_aff_col].to_numpy(float)
igg_ova=igg_binding[igg_ova_col].to_numpy(float)

AA_ORDER=np.array(sorted('ACDEFGHIKLMNPQRSTVWY'))
AA_TO_INDEX={aa:i for i,aa in enumerate(AA_ORDER.tolist())}
SEQ_LENGTH=115

def generate_onehot(sequences):
    sequences=np.asarray(sequences).astype(str)
    if any(len(s)!=SEQ_LENGTH for s in sequences):
        bad=[(i,len(s)) for i,s in enumerate(sequences) if len(s)!=SEQ_LENGTH][:5]
        raise RuntimeError(f'Unexpected sequence length: {bad}')
    out=np.zeros((len(sequences),SEQ_LENGTH,len(AA_ORDER)),dtype=np.float32)
    for i,s in enumerate(sequences):
        for j,aa in enumerate(s):
            if aa not in AA_TO_INDEX:
                raise RuntimeError(f'Noncanonical residue {aa!r} at row={i} pos={j}')
            out[i,j,AA_TO_INDEX[aa]]=1.0
    return out.reshape(len(sequences),-1)

emi_onehot=generate_onehot(emi_sequences)
iso_onehot=generate_onehot(iso_sequences)
igg_onehot=generate_onehot(igg_sequences)
assert emi_onehot.shape[1] == 2300

if 'Sample' not in igg_binding.columns:
    raise RuntimeError('IgG source lacks Sample column required for locked primary42 subset.')
igg_sample_ids=igg_binding['Sample'].astype(str).to_numpy()
counts=pd.Series(igg_sample_ids).value_counts()
bad=[x for x in IGG_PRIMARY42_SAMPLE_IDS if int(counts.get(x,0))!=1]
if bad:
    raise RuntimeError(f'IgG primary IDs missing/duplicated: {bad}')
igg_primary42_indices=np.asarray([
    int(np.flatnonzero(igg_sample_ids==x)[0]) for x in IGG_PRIMARY42_SAMPLE_IDS
],dtype=np.int64)
assert len(igg_primary42_indices)==42 and len(np.unique(igg_primary42_indices))==42

EMI_FEATURE_PATH=FEATURE_DIR/'emi_onehot.npz'
np.savez_compressed(
    EMI_FEATURE_PATH,
    sequences=emi_sequences.astype('U'),
    y_aff=y_aff_emi,
    y_ova=y_ova_emi,
    onehot=emi_onehot,
)
EXTERNAL_FEATURE_PATH=FEATURE_DIR/'external_onehot.npz'
np.savez_compressed(
    EXTERNAL_FEATURE_PATH,
    iso_n=np.asarray(len(iso_sequences),dtype=np.int64),
    igg_n=np.asarray(len(igg_sequences),dtype=np.int64),
    igg_primary42_indices=igg_primary42_indices,
    igg_primary42_sample_ids=np.asarray(IGG_PRIMARY42_SAMPLE_IDS).astype('U'),
    iso__sequences=iso_sequences.astype('U'),
    igg__sequences=igg_sequences.astype('U'),
    igg__sample_ids=igg_sample_ids.astype('U'),
    iso__y_aff=iso_aff,
    iso__y_ova=iso_ova,
    igg__y_aff=igg_aff,
    igg__y_ova=igg_ova,
    iso__onehot=iso_onehot,
    igg__onehot=igg_onehot,
)
print('EMI:', len(emi_sequences), 'ISO:', len(iso_sequences), 'IgG:', len(igg_sequences))
print('Primary42:', len(igg_primary42_indices))
print('Feature archives ready.')

## 3. Reproducibility lock
This creates a fresh lock for the downstream Pareto component experiment. It records both the eight-arm architecture suite and the isolated loss suite. The loss margins and weights are the pinned revision-code settings (`ranking margin=0.3`, `gap margin=0.2`).


In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()

LOCK = {
    'experiment':'Pareto_Component_Architecture_Ablation',
    'repository_commit':PINNED_COMMIT,
    'seeds':SEEDS,
    'features':FEATURES,
    'k_values':K_VALUES,
    'epochs':EPOCHS,
    'batch_size':BATCH_SIZE,
    'learning_rate':LEARNING_RATE,
    'architecture_arms':ARCH_ARMS,
    'loss_arms':LOSS_ARMS,
    'all_arms':ALL_ARMS,
    'arm_labels':ARM_LABELS,
    'architecture_suite':'exact prior A-H component definitions',
    'loss_suite':'focal/ranking/gap isolated controls',
    'loss_settings':EXPECTED_LOSS,
    'architecture_component_settings':ARCH_COMPONENT_SETTINGS,
    'score_space':'raw logits: maximize target, minimize OVA',
    'datasets':['ISO','IgG-primary42','IgG-all96'],
    'igg_primary42_sample_ids':IGG_PRIMARY42_SAMPLE_IDS,
}
LOCK_PATH=WORK_ROOT/'PARETO_COMPONENT_ABLATION_LOCK.json'
LOCK_PATH.write_text(json.dumps(LOCK,indent=2,sort_keys=True)+'\n',encoding='utf-8')
LOCK_SHA=sha256_file(LOCK_PATH)
(WORK_ROOT/'PARETO_COMPONENT_ABLATION_LOCK.sha256').write_text(LOCK_SHA+'\n')
print('Lock SHA256:',LOCK_SHA)

## 4. Embedded workers
Two workers are embedded so this notebook is self-contained:

1. **Architecture worker:** the exact prior A–H component-ablation implementation, now run in its existing `external` stage so all eight arms (including Full Routed-MOLM) emit ISO/IgG predictions.
2. **Loss worker:** the controlled focal/ranking/gap worker used to isolate auxiliary-loss and matched-sharing effects.

The architecture worker uses the same helper module and component definitions as the completed A–H mutation experiment.


In [ ]:
COMMON_SOURCE = 'from __future__ import annotations\n\nimport gc\nimport os\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis\nfrom sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix, f1_score, matthews_corrcoef, roc_auc_score)\n\nREPO = Path(os.environ["MOLM_REPO"]).resolve()\nSEED = int(os.environ.get("MOLM_SEED", "42"))\n# Each worker process is launched with MOLM_FEATURE_TYPE.  This value is\n# diagnostic metadata only; it does not affect model computation.\nFEATURE_TYPE = os.environ["MOLM_FEATURE_TYPE"]\nDOMINANCE_WEIGHT = float(os.environ.get("MOLM_DOMINANCE_WEIGHT", "0.03"))\nDOMINANCE_MARGIN = float(os.environ.get("MOLM_DOMINANCE_MARGIN", "0.20"))\nDOMINANCE_WARMUP = int(os.environ.get("MOLM_DOMINANCE_WARMUP", "5"))\nDOMINANCE_MAX_PAIRS = int(os.environ.get("MOLM_DOMINANCE_MAX_PAIRS", "4096"))\nPRIVATE_ADAPTER_DIM = int(os.environ.get("MOLM_PRIVATE_ADAPTER_DIM", "32"))\nPRIVATE_GATE_INIT = float(os.environ.get("MOLM_PRIVATE_GATE_INIT", "-2.0"))\nROUTING_EPS = float(os.environ.get("MOLM_ROUTING_EPS", "1e-12"))\n\nsys.path.insert(0, str(REPO))\nfrom phase0_config import (  # noqa: E402\n    DiagnosticMOLM,\n    config,\n    focal_bce_with_logits,\n    gap_hinge_loss,\n    hard_reset_rng,\n    ranking_loss,\n)\n\n# Locked final protocol.\nconfig.PARETO_LOSS = False\nconfig.EPOCHS = int(os.environ.get("MOLM_EPOCHS", "25"))\nconfig.BATCH_SIZE = int(os.environ.get("MOLM_BATCH_SIZE", "64"))\nconfig.LEARNING_RATE = float(os.environ.get("MOLM_LEARNING_RATE", "5e-5"))\n\ndef task_losses(\n    outputs,\n    y_aff,\n    y_ova,\n    aff_pos_weight,\n    ova_pos_weight,\n):\n    aff = (\n        focal_bce_with_logits(\n            y_aff,\n            outputs["aff_score"],\n            config.FOCAL_GAMMA,\n            aff_pos_weight,\n        )\n        + config.RANKING_WEIGHT_AFF\n        * ranking_loss(\n            outputs["aff_score"],\n            y_aff,\n            config.RANKING_MARGIN,\n        )\n        + config.GAP_WEIGHT_AFF\n        * gap_hinge_loss(\n            outputs["aff_score"],\n            y_aff,\n            config.GAP_MARGIN,\n        )\n    )\n    ova = (\n        focal_bce_with_logits(\n            y_ova,\n            outputs["spec_score"],\n            config.FOCAL_GAMMA,\n            ova_pos_weight,\n        )\n        + config.RANKING_WEIGHT_SPEC\n        * ranking_loss(\n            outputs["spec_score"],\n            y_ova,\n            config.RANKING_MARGIN,\n        )\n        + config.GAP_WEIGHT_SPEC\n        * gap_hinge_loss(\n            outputs["spec_score"],\n            y_ova,\n            config.GAP_MARGIN,\n        )\n    )\n    return aff, ova\n\n\ndef dominance_pair_loss(\n    aff_scores,\n    ova_scores,\n    y_aff,\n    y_ova,\n    margin=0.20,\n    max_pairs=4096,\n):\n    """\n    Coordinate-wise binary dominance:\n      target positive > target negative;\n      OVA negative > OVA positive.\n\n    Only coordinates with a strict label improvement are constrained.\n    Equal coordinates and incomparable phenotype pairs are unconstrained.\n    """\n    aff_scores = aff_scores.reshape(-1).float()\n    ova_scores = ova_scores.reshape(-1).float()\n    y_aff = y_aff.reshape(-1).float().to(aff_scores.device)\n    y_ova = y_ova.reshape(-1).float().to(aff_scores.device)\n\n    desirability = torch.stack([y_aff, 1.0 - y_ova], dim=1)\n    weakly_better = (\n        desirability[:, None, :] >= desirability[None, :, :]\n    ).all(dim=-1)\n    strictly_better = (\n        desirability[:, None, :] > desirability[None, :, :]\n    )\n    pairs = (\n        weakly_better & strictly_better.any(dim=-1)\n    ).nonzero(as_tuple=False)\n\n    if pairs.numel() == 0:\n        return aff_scores.sum() * 0.0\n\n    if len(pairs) > int(max_pairs):\n        choice = torch.randperm(\n            len(pairs),\n            device=pairs.device,\n        )[: int(max_pairs)]\n        pairs = pairs[choice]\n\n    better = pairs[:, 0]\n    worse = pairs[:, 1]\n\n    strict_aff = y_aff[better] > y_aff[worse]\n    strict_ova = (\n        (1.0 - y_ova[better])\n        > (1.0 - y_ova[worse])\n    )\n\n    terms = []\n    if strict_aff.any():\n        terms.append(\n            F.softplus(\n                float(margin)\n                - (\n                    aff_scores[better[strict_aff]]\n                    - aff_scores[worse[strict_aff]]\n                )\n            )\n        )\n    if strict_ova.any():\n        terms.append(\n            F.softplus(\n                float(margin)\n                - (\n                    ova_scores[worse[strict_ova]]\n                    - ova_scores[better[strict_ova]]\n                )\n            )\n        )\n\n    if not terms:\n        return aff_scores.sum() * 0.0\n    return torch.cat(terms).mean()\n\n\nclass LowRankPrivateAdapter(nn.Module):\n    def __init__(\n        self,\n        input_dim,\n        output_dim,\n        bottleneck_dim=32,\n        gate_init=-2.0,\n    ):\n        super().__init__()\n        self.down = nn.Linear(input_dim, bottleneck_dim)\n        self.norm = nn.LayerNorm(bottleneck_dim)\n        self.up = nn.Linear(bottleneck_dim, output_dim)\n        self.gate_logit = nn.Parameter(\n            torch.tensor(float(gate_init))\n        )\n        nn.init.normal_(\n            self.up.weight,\n            mean=0.0,\n            std=1e-3,\n        )\n        nn.init.zeros_(self.up.bias)\n\n    def forward(self, inputs):\n        correction = self.up(\n            F.gelu(\n                self.norm(\n                    self.down(inputs)\n                )\n            )\n        )\n        return (\n            torch.sigmoid(self.gate_logit)\n            * correction\n        )\n\n    def gate_value(self):\n        return float(\n            torch.sigmoid(\n                self.gate_logit.detach()\n            ).cpu()\n        )\n\n\nclass ParetoRoutedMOLM(DiagnosticMOLM):\n    def __init__(\n        self,\n        *args,\n        private_adapter_dim=32,\n        private_gate_init=-2.0,\n        **kwargs,\n    ):\n        super().__init__(*args, **kwargs)\n        shared_out_dim = (\n            self._shared_dims[-1]\n            if self._shared_dims\n            else self.input_dim\n        )\n        self.aff_private_adapter = LowRankPrivateAdapter(\n            self.input_dim,\n            shared_out_dim,\n            bottleneck_dim=private_adapter_dim,\n            gate_init=private_gate_init,\n        )\n        self.spec_private_adapter = LowRankPrivateAdapter(\n            self.input_dim,\n            shared_out_dim,\n            bottleneck_dim=private_adapter_dim,\n            gate_init=private_gate_init,\n        )\n        self.to(\n            next(self.parameters()).device\n        )\n\n    def forward(self, inputs, training=None):\n        inputs = self._coerce_input(inputs)\n        shared = self.run_shared(\n            inputs,\n            training,\n        )\n        aff_routed = (\n            shared\n            + self.aff_private_adapter(inputs)\n        )\n        spec_routed = (\n            shared\n            + self.spec_private_adapter(inputs)\n        )\n        aff_latent = self.run_tower(\n            aff_routed,\n            self.aff_layers,\n            self.aff_proj,\n            self.aff_proj_norm,\n            training,\n        )\n        spec_latent = self.run_tower(\n            spec_routed,\n            self.spec_layers,\n            self.spec_proj,\n            self.spec_proj_norm,\n            training,\n        )\n        aff_logit = self.aff_head(\n            aff_latent\n        ).squeeze(-1)\n        spec_logit = self.spec_head(\n            spec_latent\n        ).squeeze(-1)\n\n        return {\n            "aff_score": aff_logit,\n            "spec_score": spec_logit,\n            "aff_latent": aff_latent,\n            "spec_latent": spec_latent,\n            "shared": shared,\n        }\n\n    def private_gate_values(self):\n        return {\n            "aff_private_gate": (\n                self.aff_private_adapter.gate_value()\n            ),\n            "ova_private_gate": (\n                self.spec_private_adapter.gate_value()\n            ),\n        }\n\n\nclass IndependentTaskNet(nn.Module):\n    """Independent neural baseline for one binary task."""\n\n    def __init__(self, input_dim):\n        super().__init__()\n        dims = (\n            list(config.SHARED_DIMS)\n            + list(config.TOWER_DIMS)\n        )\n        self.blocks = DiagnosticMOLM._make_blocks(\n            input_dim,\n            dims,\n            config.DROPOUT_RATE,\n        )\n        output_dim = (\n            dims[-1]\n            if dims\n            else input_dim\n        )\n        self.proj = nn.Linear(\n            output_dim,\n            config.LATENT_DIM,\n        )\n        self.proj_norm = nn.LayerNorm(\n            config.LATENT_DIM\n        )\n        self.head = nn.Linear(\n            config.LATENT_DIM,\n            1,\n        )\n\n    @staticmethod\n    def _dropout(layer, inputs, training):\n        if training is None:\n            return layer(inputs)\n        return F.dropout(\n            inputs,\n            p=layer.p,\n            training=training,\n        )\n\n    def forward(self, inputs, training=None):\n        for index in range(\n            0,\n            len(self.blocks),\n            3,\n        ):\n            inputs = self.blocks[index](inputs)\n            inputs = self.blocks[index + 1](inputs)\n            inputs = F.gelu(inputs)\n            inputs = self._dropout(\n                self.blocks[index + 2],\n                inputs,\n                training,\n            )\n        latent = self.proj_norm(\n            self.proj(inputs)\n        )\n        score = self.head(\n            latent\n        ).squeeze(-1)\n        return score\n\n\nclass IndependentNNPair(nn.Module):\n    """Two independently parameterized task networks."""\n\n    def __init__(self, input_dim):\n        super().__init__()\n        self.aff_net = IndependentTaskNet(input_dim)\n        self.ova_net = IndependentTaskNet(input_dim)\n\n    def forward(self, inputs, training=None):\n        return {\n            "aff_score": self.aff_net(\n                inputs,\n                training=training,\n            ),\n            "spec_score": self.ova_net(\n                inputs,\n                training=training,\n            ),\n        }\n\n\nclass RoutedTrainer:\n    def __init__(\n        self,\n        model,\n        aff_pos_weight,\n        ova_pos_weight,\n    ):\n        self.device = torch.device(\n            "cuda"\n            if torch.cuda.is_available()\n            else "cpu"\n        )\n        self.model = model.to(self.device)\n        self.aff_pos_weight = float(\n            aff_pos_weight\n        )\n        self.ova_pos_weight = float(\n            ova_pos_weight\n        )\n        self.optimizer = torch.optim.AdamW(\n            self.model.parameters(),\n            lr=config.LEARNING_RATE,\n            weight_decay=1e-4,\n        )\n        self.rows = []\n\n    @staticmethod\n    def _replace_none(\n        gradients,\n        parameters,\n    ):\n        return [\n            (\n                torch.zeros_like(parameter)\n                if gradient is None\n                else gradient\n            )\n            for gradient, parameter\n            in zip(\n                gradients,\n                parameters,\n            )\n        ]\n\n    @staticmethod\n    def _dot(first, second):\n        return sum(\n            (left * right).sum()\n            for left, right\n            in zip(first, second)\n        )\n\n    @staticmethod\n    def _norm_squared(gradients):\n        return sum(\n            (gradient * gradient).sum()\n            for gradient in gradients\n        )\n\n    def fit(\n        self,\n        X,\n        y_aff,\n        y_ova,\n        label,\n    ):\n        dataset = (\n            torch.utils.data.TensorDataset(\n                torch.as_tensor(\n                    X,\n                    dtype=torch.float32,\n                ),\n                torch.as_tensor(\n                    y_aff,\n                    dtype=torch.float32,\n                ),\n                torch.as_tensor(\n                    y_ova,\n                    dtype=torch.float32,\n                ),\n            )\n        )\n        generator = (\n            torch.Generator().manual_seed(SEED)\n        )\n        loader = torch.utils.data.DataLoader(\n            dataset,\n            batch_size=config.BATCH_SIZE,\n            shuffle=True,\n            generator=generator,\n        )\n\n        shared_parameters = [\n            parameter\n            for parameter\n            in self.model.shared_layers.parameters()\n            if parameter.requires_grad\n        ]\n\n        for epoch in range(config.EPOCHS):\n            dominance_active = (\n                DOMINANCE_WEIGHT > 0\n                and epoch >= DOMINANCE_WARMUP\n            )\n            totals = []\n            dominance_values = []\n            conflicts = []\n            pre_cosines = []\n            post_cosines = []\n\n            for (\n                X_batch,\n                y_aff_batch,\n                y_ova_batch,\n            ) in loader:\n                X_batch = X_batch.to(self.device)\n                y_aff_batch = y_aff_batch.to(\n                    self.device\n                )\n                y_ova_batch = y_ova_batch.to(\n                    self.device\n                )\n\n                self.model.train()\n                self.optimizer.zero_grad(\n                    set_to_none=True\n                )\n\n                outputs = self.model(\n                    X_batch,\n                    training=True,\n                )\n                aff_loss, ova_loss = task_losses(\n                    outputs,\n                    y_aff_batch,\n                    y_ova_batch,\n                    self.aff_pos_weight,\n                    self.ova_pos_weight,\n                )\n                dominance = dominance_pair_loss(\n                    outputs["aff_score"],\n                    outputs["spec_score"],\n                    y_aff_batch,\n                    y_ova_batch,\n                    margin=DOMINANCE_MARGIN,\n                    max_pairs=DOMINANCE_MAX_PAIRS,\n                )\n\n                aff_gradients = torch.autograd.grad(\n                    aff_loss,\n                    shared_parameters,\n                    retain_graph=True,\n                    allow_unused=True,\n                )\n                ova_gradients = torch.autograd.grad(\n                    ova_loss,\n                    shared_parameters,\n                    retain_graph=True,\n                    allow_unused=True,\n                )\n                aff_gradients = self._replace_none(\n                    aff_gradients,\n                    shared_parameters,\n                )\n                ova_gradients = self._replace_none(\n                    ova_gradients,\n                    shared_parameters,\n                )\n\n                if dominance_active:\n                    dominance_gradients = (\n                        torch.autograd.grad(\n                            dominance,\n                            shared_parameters,\n                            retain_graph=True,\n                            allow_unused=True,\n                        )\n                    )\n                    dominance_gradients = (\n                        self._replace_none(\n                            dominance_gradients,\n                            shared_parameters,\n                        )\n                    )\n                else:\n                    dominance_gradients = [\n                        torch.zeros_like(parameter)\n                        for parameter\n                        in shared_parameters\n                    ]\n\n                dot_product = self._dot(\n                    aff_gradients,\n                    ova_gradients,\n                )\n                aff_norm = self._norm_squared(\n                    aff_gradients\n                )\n                ova_norm = self._norm_squared(\n                    ova_gradients\n                )\n                pre_cosine = dot_product / (\n                    torch.sqrt(\n                        aff_norm * ova_norm\n                    )\n                    + ROUTING_EPS\n                )\n                conflict = bool(\n                    dot_product.detach().cpu() < 0\n                )\n\n                if conflict:\n                    aff_coefficient = (\n                        dot_product\n                        / (\n                            ova_norm\n                            + ROUTING_EPS\n                        )\n                    )\n                    ova_coefficient = (\n                        dot_product\n                        / (\n                            aff_norm\n                            + ROUTING_EPS\n                        )\n                    )\n                    routed_aff = [\n                        gradient_aff\n                        - aff_coefficient\n                        * gradient_ova\n                        for gradient_aff, gradient_ova\n                        in zip(\n                            aff_gradients,\n                            ova_gradients,\n                        )\n                    ]\n                    routed_ova = [\n                        gradient_ova\n                        - ova_coefficient\n                        * gradient_aff\n                        for gradient_aff, gradient_ova\n                        in zip(\n                            aff_gradients,\n                            ova_gradients,\n                        )\n                    ]\n                else:\n                    routed_aff = aff_gradients\n                    routed_ova = ova_gradients\n\n                post_dot = self._dot(\n                    routed_aff,\n                    routed_ova,\n                )\n                post_aff_norm = self._norm_squared(\n                    routed_aff\n                )\n                post_ova_norm = self._norm_squared(\n                    routed_ova\n                )\n                post_cosine = post_dot / (\n                    torch.sqrt(\n                        post_aff_norm\n                        * post_ova_norm\n                    )\n                    + ROUTING_EPS\n                )\n\n                total = aff_loss + ova_loss\n                if dominance_active:\n                    total = (\n                        total\n                        + DOMINANCE_WEIGHT\n                        * dominance\n                    )\n\n                total.backward()\n\n                for (\n                    parameter,\n                    gradient_aff,\n                    gradient_ova,\n                    gradient_dominance,\n                ) in zip(\n                    shared_parameters,\n                    routed_aff,\n                    routed_ova,\n                    dominance_gradients,\n                ):\n                    routed_gradient = (\n                        gradient_aff\n                        + gradient_ova\n                    )\n                    if dominance_active:\n                        routed_gradient = (\n                            routed_gradient\n                            + DOMINANCE_WEIGHT\n                            * gradient_dominance\n                        )\n                    parameter.grad = (\n                        routed_gradient.detach().clone()\n                    )\n\n                torch.nn.utils.clip_grad_norm_(\n                    self.model.parameters(),\n                    1.0,\n                )\n                self.optimizer.step()\n\n                totals.append(\n                    float(total.detach().cpu())\n                )\n                dominance_values.append(\n                    float(\n                        dominance.detach().cpu()\n                    )\n                )\n                conflicts.append(float(conflict))\n                pre_cosines.append(\n                    float(\n                        pre_cosine.detach().cpu()\n                    )\n                )\n                post_cosines.append(\n                    float(\n                        post_cosine.detach().cpu()\n                    )\n                )\n\n            gates = (\n                self.model.private_gate_values()\n            )\n            row = {\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "model": "Routed-MOLM",\n                "epoch": epoch + 1,\n                "dominance_active": (\n                    dominance_active\n                ),\n                "mean_total_loss": float(\n                    np.mean(totals)\n                ),\n                "mean_dominance_loss": float(\n                    np.mean(\n                        dominance_values\n                    )\n                ),\n                "conflict_rate": float(\n                    np.mean(conflicts)\n                ),\n                "mean_pre_route_cosine": float(\n                    np.mean(pre_cosines)\n                ),\n                "mean_post_route_cosine": float(\n                    np.mean(post_cosines)\n                ),\n                **gates,\n            }\n            self.rows.append(row)\n\n            if (\n                epoch == 0\n                or (epoch + 1) % 5 == 0\n            ):\n                print(\n                    f"{label} epoch={epoch+1:02d}/"\n                    f"{config.EPOCHS} "\n                    f"loss={row[\'mean_total_loss\']:.4f} "\n                    f"conflict={row[\'conflict_rate\']:.3f}",\n                    flush=True,\n                )\n\n        return self\n\n\nclass NNTrainer:\n    def __init__(\n        self,\n        model,\n        aff_pos_weight,\n        ova_pos_weight,\n    ):\n        self.device = torch.device(\n            "cuda"\n            if torch.cuda.is_available()\n            else "cpu"\n        )\n        self.model = model.to(self.device)\n        self.aff_pos_weight = float(\n            aff_pos_weight\n        )\n        self.ova_pos_weight = float(\n            ova_pos_weight\n        )\n        self.optimizer = torch.optim.AdamW(\n            self.model.parameters(),\n            lr=config.LEARNING_RATE,\n            weight_decay=1e-4,\n        )\n        self.rows = []\n\n    def fit(\n        self,\n        X,\n        y_aff,\n        y_ova,\n        label,\n    ):\n        dataset = (\n            torch.utils.data.TensorDataset(\n                torch.as_tensor(\n                    X,\n                    dtype=torch.float32,\n                ),\n                torch.as_tensor(\n                    y_aff,\n                    dtype=torch.float32,\n                ),\n                torch.as_tensor(\n                    y_ova,\n                    dtype=torch.float32,\n                ),\n            )\n        )\n        generator = (\n            torch.Generator().manual_seed(\n                SEED + 100000\n            )\n        )\n        loader = torch.utils.data.DataLoader(\n            dataset,\n            batch_size=config.BATCH_SIZE,\n            shuffle=True,\n            generator=generator,\n        )\n\n        for epoch in range(config.EPOCHS):\n            totals = []\n            for (\n                X_batch,\n                y_aff_batch,\n                y_ova_batch,\n            ) in loader:\n                X_batch = X_batch.to(self.device)\n                y_aff_batch = y_aff_batch.to(\n                    self.device\n                )\n                y_ova_batch = y_ova_batch.to(\n                    self.device\n                )\n\n                self.model.train()\n                self.optimizer.zero_grad(\n                    set_to_none=True\n                )\n                outputs = self.model(\n                    X_batch,\n                    training=True,\n                )\n                aff_loss = focal_bce_with_logits(\n                    y_aff_batch,\n                    outputs["aff_score"],\n                    config.FOCAL_GAMMA,\n                    self.aff_pos_weight,\n                )\n                ova_loss = focal_bce_with_logits(\n                    y_ova_batch,\n                    outputs["spec_score"],\n                    config.FOCAL_GAMMA,\n                    self.ova_pos_weight,\n                )\n                total = aff_loss + ova_loss\n                total.backward()\n                torch.nn.utils.clip_grad_norm_(\n                    self.model.parameters(),\n                    1.0,\n                )\n                self.optimizer.step()\n                totals.append(\n                    float(total.detach().cpu())\n                )\n\n            row = {\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "model": "NN",\n                "epoch": epoch + 1,\n                "dominance_active": False,\n                "mean_total_loss": float(\n                    np.mean(totals)\n                ),\n                "mean_dominance_loss": np.nan,\n                "conflict_rate": np.nan,\n                "mean_pre_route_cosine": np.nan,\n                "mean_post_route_cosine": np.nan,\n                "aff_private_gate": np.nan,\n                "ova_private_gate": np.nan,\n            }\n            self.rows.append(row)\n\n            if (\n                epoch == 0\n                or (epoch + 1) % 5 == 0\n            ):\n                print(\n                    f"{label} epoch={epoch+1:02d}/"\n                    f"{config.EPOCHS} "\n                    f"loss={row[\'mean_total_loss\']:.4f}",\n                    flush=True,\n                )\n\n        return self\n\n\ndef make_routed(input_dim):\n    return ParetoRoutedMOLM(\n        input_dim=input_dim,\n        latent_dim=config.LATENT_DIM,\n        shared_dims=config.SHARED_DIMS,\n        tower_dims=config.TOWER_DIMS,\n        dropout_rate=config.DROPOUT_RATE,\n        grl_lambda=config.GRL_LAMBDA,\n        private_adapter_dim=PRIVATE_ADAPTER_DIM,\n        private_gate_init=PRIVATE_GATE_INIT,\n    )\n\n\ndef predict_torch_model(model, X):\n    model.eval()\n    device = next(model.parameters()).device\n    with torch.no_grad():\n        outputs = model(\n            torch.as_tensor(\n                X,\n                dtype=torch.float32,\n                device=device,\n            ),\n            training=False,\n        )\n    return (\n        outputs["aff_score"]\n        .detach()\n        .cpu()\n        .numpy()\n        .reshape(-1),\n        outputs["spec_score"]\n        .detach()\n        .cpu()\n        .numpy()\n        .reshape(-1),\n    )\n\n\ndef binary_metrics(y_true, score):\n    y_true = np.asarray(\n        y_true,\n        dtype=int,\n    ).reshape(-1)\n    score = np.asarray(\n        score,\n        dtype=float,\n    ).reshape(-1)\n    y_pred = (score >= 0.0).astype(int)\n\n    tn, fp, fn, tp = confusion_matrix(\n        y_true,\n        y_pred,\n        labels=[0, 1],\n    ).ravel()\n\n    sensitivity = (\n        tp / (tp + fn)\n        if (tp + fn)\n        else np.nan\n    )\n    specificity = (\n        tn / (tn + fp)\n        if (tn + fp)\n        else np.nan\n    )\n    npv = (\n        tn / (tn + fn)\n        if (tn + fn)\n        else np.nan\n    )\n\n    if len(np.unique(y_true)) == 2:\n        auroc = roc_auc_score(\n            y_true,\n            score,\n        )\n        auprc = average_precision_score(\n            y_true,\n            score,\n        )\n    else:\n        auroc = np.nan\n        auprc = np.nan\n\n    return {\n        "n": int(len(y_true)),\n        "n_negative": int((y_true == 0).sum()),\n        "n_positive": int((y_true == 1).sum()),\n        "accuracy": float(\n            accuracy_score(y_true, y_pred)\n        ),\n        "balanced_accuracy": float(\n            balanced_accuracy_score(\n                y_true,\n                y_pred,\n            )\n        ),\n        "mcc": float(\n            matthews_corrcoef(\n                y_true,\n                y_pred,\n            )\n        ),\n        "f1": float(\n            f1_score(\n                y_true,\n                y_pred,\n                zero_division=0,\n            )\n        ),\n        "sensitivity": float(sensitivity),\n        "specificity": float(specificity),\n        "npv": float(npv),\n        "auroc": float(auroc),\n        "auprc": float(auprc),\n        "tn": int(tn),\n        "fp": int(fp),\n        "fn": int(fn),\n        "tp": int(tp),\n    }\n\n\n\ndef train_lda_pair(X_train, y_aff_train, y_ova_train):\n    aff = LinearDiscriminantAnalysis(solver="svd")\n    ova = LinearDiscriminantAnalysis(solver="svd")\n    aff.fit(X_train, np.asarray(y_aff_train, dtype=int))\n    ova.fit(X_train, np.asarray(y_ova_train, dtype=int))\n    return aff, ova\n\n\ndef predict_lda_pair(pair, X):\n    aff, ova = pair\n    return (\n        np.asarray(aff.decision_function(X), dtype=float).reshape(-1),\n        np.asarray(ova.decision_function(X), dtype=float).reshape(-1),\n    )\n\n\ndef fit_routed(X_train, y_aff_train, y_ova_train, seed, label="Routed-MOLM"):\n    os.environ["MOLM_SEED"] = str(seed)\n    hard_reset_rng(int(seed), label)\n    aff_pos_weight = float((np.asarray(y_aff_train) == 0).sum() / max(int((np.asarray(y_aff_train) == 1).sum()), 1))\n    ova_pos_weight = float((np.asarray(y_ova_train) == 0).sum() / max(int((np.asarray(y_ova_train) == 1).sum()), 1))\n    model = make_routed(X_train.shape[1])\n    trainer = RoutedTrainer(model, aff_pos_weight, ova_pos_weight)\n    trainer.fit(X_train, y_aff_train, y_ova_train, label)\n    return model, trainer\n\n\ndef fit_nn(X_train, y_aff_train, y_ova_train, seed, label="NN"):\n    os.environ["MOLM_SEED"] = str(seed)\n    hard_reset_rng(int(seed) + 100000, label)\n    aff_pos_weight = float((np.asarray(y_aff_train) == 0).sum() / max(int((np.asarray(y_aff_train) == 1).sum()), 1))\n    ova_pos_weight = float((np.asarray(y_ova_train) == 0).sum() / max(int((np.asarray(y_ova_train) == 1).sum()), 1))\n    model = IndependentNNPair(X_train.shape[1])\n    trainer = NNTrainer(model, aff_pos_weight, ova_pos_weight)\n    trainer.fit(X_train, y_aff_train, y_ova_train, label)\n    return model, trainer\n'
ARCH_WORKER_SOURCE = 'from __future__ import annotations\n\nimport gc\nimport hashlib\nimport json\nimport os\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom scipy.stats import spearmanr\n\n# Required env must be set before importing the shared helper.\nREPO = Path(os.environ["MOLM_REPO"]).resolve()\nFEATURE_TYPE = os.environ["MOLM_FEATURE_TYPE"]\nSEED = int(os.environ.get("MOLM_SEED", "42"))\nSTAGE = os.environ.get("MOLM_STAGE", "mutation")\nARM_SET = os.environ.get("MOLM_ARM_SET", "core")\nOUT_DIR = Path(os.environ["MOLM_JOB_OUTPUT"]).resolve()\nEMI_FEATURE_PATH = Path(os.environ["MOLM_EMI_FEATURE_PATH"]).resolve()\nEXTERNAL_FEATURE_PATH = Path(os.environ.get("MOLM_EXTERNAL_FEATURE_PATH", "")).resolve() if os.environ.get("MOLM_EXTERNAL_FEATURE_PATH") else None\nHOLDOUT_PATH = Path(os.environ.get("MOLM_HOLDOUT_PATH", "")).resolve() if os.environ.get("MOLM_HOLDOUT_PATH") else None\nLOCK_HASH = os.environ["MOLM_LOCK_HASH"]\nSMOKE = os.environ.get("MOLM_SMOKE", "0") == "1"\n\nOUT_DIR.mkdir(parents=True, exist_ok=True)\n\nfrom molm_definitive_common import (  # noqa: E402\n    DOMINANCE_MARGIN,\n    DOMINANCE_MAX_PAIRS,\n    DOMINANCE_WARMUP,\n    DOMINANCE_WEIGHT,\n    PRIVATE_ADAPTER_DIM,\n    PRIVATE_GATE_INIT,\n    ROUTING_EPS,\n    IndependentNNPair,\n    LowRankPrivateAdapter,\n    binary_metrics,\n    config,\n    dominance_pair_loss,\n    hard_reset_rng,\n    predict_torch_model,\n    task_losses,\n)\n\n# Pinned repository model.\nimport sys\nsys.path.insert(0, str(REPO))\nfrom phase0_config import DiagnosticMOLM  # noqa: E402\n\n\n@dataclass(frozen=True)\nclass ArmSpec:\n    name: str\n    architecture: str  # shared | private | capacity_shared | independent\n    dominance: bool\n    pcgrad: bool\n    label: str\n\n\nARM_SPECS = {\n    "shared_base": ArmSpec(\n        "shared_base", "shared", False, False,\n        "A Shared Base MOLM",\n    ),\n    "shared_dom": ArmSpec(\n        "shared_dom", "shared", True, False,\n        "B Shared + Dominance",\n    ),\n    "shared_dom_pcgrad": ArmSpec(\n        "shared_dom_pcgrad", "shared", True, True,\n        "C Shared + Dominance + PCGrad",\n    ),\n    "shared_dom_private": ArmSpec(\n        "shared_dom_private", "private", True, False,\n        "D Shared + Dominance + Private",\n    ),\n    "full_routed": ArmSpec(\n        "full_routed", "private", True, True,\n        "E Full Routed-MOLM",\n    ),\n    "independent_st_dom": ArmSpec(\n        "independent_st_dom", "independent", True, False,\n        "F Independent-parameter + matched joint loss",\n    ),\n    "shared_capacity_dom": ArmSpec(\n        "shared_capacity_dom", "capacity_shared", True, False,\n        "G Capacity-Matched Shared + Dominance",\n    ),\n    "shared_capacity_dom_pcgrad": ArmSpec(\n        "shared_capacity_dom_pcgrad", "capacity_shared", True, True,\n        "H Capacity-Matched Shared + Dominance + PCGrad",\n    ),\n}\n\nCORE_ARMS = [\n    "shared_base",\n    "shared_dom",\n    "shared_dom_pcgrad",\n    "shared_dom_private",\n    "full_routed",\n    "independent_st_dom",\n    "shared_capacity_dom",\n    "shared_capacity_dom_pcgrad",\n]\nSITE_CONTROL_ARMS = [\n    "shared_dom",\n    "full_routed",\n    "independent_st_dom",\n]\n\nif ARM_SET == "core":\n    ARMS = CORE_ARMS\nelif ARM_SET == "site_control":\n    ARMS = SITE_CONTROL_ARMS\nelif ARM_SET == "smoke":\n    ARMS = CORE_ARMS\nelse:\n    raise ValueError(f"Unknown ARM_SET={ARM_SET!r}")\n\n\nclass PrivateAblationMOLM(DiagnosticMOLM):\n    """Task-private low-rank corrections on top of the same shared encoder."""\n\n    def __init__(\n        self,\n        *args,\n        private_adapter_dim=32,\n        private_gate_init=-2.0,\n        **kwargs,\n    ):\n        super().__init__(*args, **kwargs)\n        shared_out_dim = self._shared_dims[-1] if self._shared_dims else self.input_dim\n        self.aff_private_adapter = LowRankPrivateAdapter(\n            self.input_dim,\n            shared_out_dim,\n            bottleneck_dim=private_adapter_dim,\n            gate_init=private_gate_init,\n        )\n        self.spec_private_adapter = LowRankPrivateAdapter(\n            self.input_dim,\n            shared_out_dim,\n            bottleneck_dim=private_adapter_dim,\n            gate_init=private_gate_init,\n        )\n\n    def forward(self, inputs, training=None):\n        inputs = self._coerce_input(inputs)\n        shared = self.run_shared(inputs, training)\n        aff_private = self.aff_private_adapter(inputs)\n        ova_private = self.spec_private_adapter(inputs)\n        aff_routed = shared + aff_private\n        ova_routed = shared + ova_private\n\n        aff_latent = self.run_tower(\n            aff_routed,\n            self.aff_layers,\n            self.aff_proj,\n            self.aff_proj_norm,\n            training,\n        )\n        ova_latent = self.run_tower(\n            ova_routed,\n            self.spec_layers,\n            self.spec_proj,\n            self.spec_proj_norm,\n            training,\n        )\n        aff_score = self.aff_head(aff_latent).squeeze(-1)\n        ova_score = self.spec_head(ova_latent).squeeze(-1)\n        return {\n            "aff_score": aff_score,\n            "spec_score": ova_score,\n            "aff_latent": aff_latent,\n            "spec_latent": ova_latent,\n            "shared": shared,\n            "aff_private": aff_private,\n            "ova_private": ova_private,\n        }\n\n    def private_gate_values(self):\n        return {\n            "aff_private_gate": self.aff_private_adapter.gate_value(),\n            "ova_private_gate": self.spec_private_adapter.gate_value(),\n        }\n\n\nclass CapacityMatchedSharedMOLM(DiagnosticMOLM):\n    """\n    Parameter-count control.\n\n    Two adapters with the same shapes as the task-private arm are present, but\n    BOTH tasks receive the same averaged correction. Therefore this adds\n    approximately the same adapter parameter budget without task-private routing.\n    """\n\n    def __init__(\n        self,\n        *args,\n        private_adapter_dim=32,\n        private_gate_init=-2.0,\n        **kwargs,\n    ):\n        super().__init__(*args, **kwargs)\n        shared_out_dim = self._shared_dims[-1] if self._shared_dims else self.input_dim\n        self.capacity_adapter_1 = LowRankPrivateAdapter(\n            self.input_dim,\n            shared_out_dim,\n            bottleneck_dim=private_adapter_dim,\n            gate_init=private_gate_init,\n        )\n        self.capacity_adapter_2 = LowRankPrivateAdapter(\n            self.input_dim,\n            shared_out_dim,\n            bottleneck_dim=private_adapter_dim,\n            gate_init=private_gate_init,\n        )\n\n    def forward(self, inputs, training=None):\n        inputs = self._coerce_input(inputs)\n        shared = self.run_shared(inputs, training)\n        correction_1 = self.capacity_adapter_1(inputs)\n        correction_2 = self.capacity_adapter_2(inputs)\n        shared_correction = 0.5 * (correction_1 + correction_2)\n        routed = shared + shared_correction\n\n        aff_latent = self.run_tower(\n            routed,\n            self.aff_layers,\n            self.aff_proj,\n            self.aff_proj_norm,\n            training,\n        )\n        ova_latent = self.run_tower(\n            routed,\n            self.spec_layers,\n            self.spec_proj,\n            self.spec_proj_norm,\n            training,\n        )\n        return {\n            "aff_score": self.aff_head(aff_latent).squeeze(-1),\n            "spec_score": self.spec_head(ova_latent).squeeze(-1),\n            "aff_latent": aff_latent,\n            "spec_latent": ova_latent,\n            "shared": shared,\n            "shared_capacity_correction": shared_correction,\n        }\n\n    def capacity_gate_values(self):\n        return {\n            "capacity_gate_1": self.capacity_adapter_1.gate_value(),\n            "capacity_gate_2": self.capacity_adapter_2.gate_value(),\n        }\n\n\ndef make_model(input_dim: int, spec: ArmSpec):\n    kwargs = dict(\n        input_dim=input_dim,\n        latent_dim=config.LATENT_DIM,\n        shared_dims=config.SHARED_DIMS,\n        tower_dims=config.TOWER_DIMS,\n        dropout_rate=config.DROPOUT_RATE,\n        grl_lambda=config.GRL_LAMBDA,\n    )\n    if spec.architecture == "shared":\n        return DiagnosticMOLM(**kwargs)\n    if spec.architecture == "private":\n        return PrivateAblationMOLM(\n            **kwargs,\n            private_adapter_dim=PRIVATE_ADAPTER_DIM,\n            private_gate_init=PRIVATE_GATE_INIT,\n        )\n    if spec.architecture == "capacity_shared":\n        return CapacityMatchedSharedMOLM(\n            **kwargs,\n            private_adapter_dim=PRIVATE_ADAPTER_DIM,\n            private_gate_init=PRIVATE_GATE_INIT,\n        )\n    if spec.architecture == "independent":\n        return IndependentNNPair(input_dim)\n    raise ValueError(spec.architecture)\n\n\ndef trainable_parameter_count(model):\n    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))\n\n\ndef _replace_none(grads, params):\n    return [\n        torch.zeros_like(p) if g is None else g\n        for g, p in zip(grads, params)\n    ]\n\n\ndef _dot(a, b):\n    if not a:\n        return torch.tensor(float("nan"))\n    return sum((x * y).sum() for x, y in zip(a, b))\n\n\ndef _norm_sq(a):\n    if not a:\n        return torch.tensor(float("nan"))\n    return sum((x * x).sum() for x in a)\n\n\ndef _cosine(a, b):\n    dot = _dot(a, b)\n    if not torch.isfinite(dot):\n        return torch.tensor(float("nan"), device=a[0].device if a else "cpu")\n    return dot / (torch.sqrt(_norm_sq(a) * _norm_sq(b)) + ROUTING_EPS)\n\n\ndef _grad_norm(a):\n    if not a:\n        return float("nan")\n    value = torch.sqrt(_norm_sq(a))\n    return float(value.detach().cpu())\n\n\ndef _ratio_norm(numerator, denominator):\n    num = numerator.norm(dim=1)\n    den = denominator.norm(dim=1).clamp_min(1e-12)\n    return float((num / den).mean().detach().cpu())\n\n\nclass ControlledTrainer:\n    def __init__(self, model, spec, aff_pos_weight, ova_pos_weight):\n        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n        self.model = model.to(self.device)\n        self.spec = spec\n        self.aff_pos_weight = float(aff_pos_weight)\n        self.ova_pos_weight = float(ova_pos_weight)\n        self.optimizer = torch.optim.AdamW(\n            self.model.parameters(),\n            lr=config.LEARNING_RATE,\n            weight_decay=1e-4,\n        )\n        self.rows = []\n\n    def fit(self, X, y_aff, y_ova, label):\n        dataset = torch.utils.data.TensorDataset(\n            torch.as_tensor(X, dtype=torch.float32),\n            torch.as_tensor(y_aff, dtype=torch.float32),\n            torch.as_tensor(y_ova, dtype=torch.float32),\n        )\n        generator = torch.Generator().manual_seed(SEED)\n        loader = torch.utils.data.DataLoader(\n            dataset,\n            batch_size=config.BATCH_SIZE,\n            shuffle=True,\n            generator=generator,\n        )\n\n        has_shared = self.spec.architecture != "independent"\n        shared_params = (\n            [\n                p for p in self.model.shared_layers.parameters()\n                if p.requires_grad\n            ]\n            if has_shared else []\n        )\n\n        for epoch in range(config.EPOCHS):\n            dom_active = bool(\n                self.spec.dominance\n                and DOMINANCE_WEIGHT > 0\n                and epoch >= DOMINANCE_WARMUP\n            )\n            totals, dom_values = [], []\n            conflicts, pre_cos, post_cos = [], [], []\n            aff_norms, ova_norms, proj_fracs = [], [], []\n            private_aff_ratios, private_ova_ratios = [], []\n            capacity_ratios = []\n\n            epoch_aff_sum = [\n                torch.zeros_like(p, device=self.device) for p in shared_params\n            ]\n            epoch_ova_sum = [\n                torch.zeros_like(p, device=self.device) for p in shared_params\n            ]\n\n            for Xb, yab, yob in loader:\n                Xb = Xb.to(self.device)\n                yab = yab.to(self.device)\n                yob = yob.to(self.device)\n\n                self.model.train()\n                self.optimizer.zero_grad(set_to_none=True)\n\n                outputs = self.model(Xb, training=True)\n                aff_loss, ova_loss = task_losses(\n                    outputs,\n                    yab,\n                    yob,\n                    self.aff_pos_weight,\n                    self.ova_pos_weight,\n                )\n\n                if self.spec.dominance:\n                    dom = dominance_pair_loss(\n                        outputs["aff_score"],\n                        outputs["spec_score"],\n                        yab,\n                        yob,\n                        margin=DOMINANCE_MARGIN,\n                        max_pairs=DOMINANCE_MAX_PAIRS,\n                    )\n                else:\n                    dom = outputs["aff_score"].sum() * 0.0\n\n                routed_aff = routed_ova = None\n                dom_grads = None\n\n                if has_shared:\n                    aff_grads = _replace_none(\n                        torch.autograd.grad(\n                            aff_loss,\n                            shared_params,\n                            retain_graph=True,\n                            allow_unused=True,\n                        ),\n                        shared_params,\n                    )\n                    ova_grads = _replace_none(\n                        torch.autograd.grad(\n                            ova_loss,\n                            shared_params,\n                            retain_graph=True,\n                            allow_unused=True,\n                        ),\n                        shared_params,\n                    )\n\n                    for acc, grad in zip(epoch_aff_sum, aff_grads):\n                        acc.add_(grad.detach())\n                    for acc, grad in zip(epoch_ova_sum, ova_grads):\n                        acc.add_(grad.detach())\n\n                    dot = _dot(aff_grads, ova_grads)\n                    conflict = bool(float(dot.detach().cpu()) < 0.0)\n                    pre = _cosine(aff_grads, ova_grads)\n\n                    if self.spec.pcgrad and conflict:\n                        aff_norm_sq = _norm_sq(aff_grads)\n                        ova_norm_sq = _norm_sq(ova_grads)\n                        aff_coeff = dot / (ova_norm_sq + ROUTING_EPS)\n                        ova_coeff = dot / (aff_norm_sq + ROUTING_EPS)\n                        routed_aff = [\n                            ga - aff_coeff * go\n                            for ga, go in zip(aff_grads, ova_grads)\n                        ]\n                        routed_ova = [\n                            go - ova_coeff * ga\n                            for ga, go in zip(aff_grads, ova_grads)\n                        ]\n                    else:\n                        routed_aff = aff_grads\n                        routed_ova = ova_grads\n\n                    post = _cosine(routed_aff, routed_ova)\n\n                    original_norm = torch.sqrt(\n                        _norm_sq(aff_grads) + _norm_sq(ova_grads)\n                    )\n                    projection_norm = torch.sqrt(\n                        sum(\n                            ((ra - ga) ** 2).sum()\n                            for ra, ga in zip(routed_aff, aff_grads)\n                        )\n                        + sum(\n                            ((ro - go) ** 2).sum()\n                            for ro, go in zip(routed_ova, ova_grads)\n                        )\n                    )\n                    projection_fraction = float(\n                        (projection_norm / (original_norm + ROUTING_EPS))\n                        .detach().cpu()\n                    )\n\n                    if self.spec.pcgrad and dom_active:\n                        dom_grads = _replace_none(\n                            torch.autograd.grad(\n                                dom,\n                                shared_params,\n                                retain_graph=True,\n                                allow_unused=True,\n                            ),\n                            shared_params,\n                        )\n\n                    conflicts.append(float(conflict))\n                    pre_cos.append(float(pre.detach().cpu()))\n                    post_cos.append(float(post.detach().cpu()))\n                    aff_norms.append(_grad_norm(aff_grads))\n                    ova_norms.append(_grad_norm(ova_grads))\n                    proj_fracs.append(projection_fraction)\n\n                total = aff_loss + ova_loss\n                if dom_active:\n                    total = total + DOMINANCE_WEIGHT * dom\n\n                total.backward()\n\n                # For PCGrad arms only, overwrite shared gradients with the\n                # routed task gradients + ordinary dominance gradient.\n                if has_shared and self.spec.pcgrad:\n                    for index, param in enumerate(shared_params):\n                        combined = routed_aff[index] + routed_ova[index]\n                        if dom_active:\n                            combined = (\n                                combined\n                                + DOMINANCE_WEIGHT * dom_grads[index]\n                            )\n                        param.grad = combined.detach().clone()\n\n                torch.nn.utils.clip_grad_norm_(\n                    self.model.parameters(),\n                    max_norm=1.0,\n                )\n                self.optimizer.step()\n\n                totals.append(float(total.detach().cpu()))\n                dom_values.append(float(dom.detach().cpu()))\n\n                if "aff_private" in outputs:\n                    private_aff_ratios.append(\n                        _ratio_norm(outputs["aff_private"], outputs["shared"])\n                    )\n                    private_ova_ratios.append(\n                        _ratio_norm(outputs["ova_private"], outputs["shared"])\n                    )\n                if "shared_capacity_correction" in outputs:\n                    capacity_ratios.append(\n                        _ratio_norm(\n                            outputs["shared_capacity_correction"],\n                            outputs["shared"],\n                        )\n                    )\n\n            epoch_global_cos = (\n                float(_cosine(epoch_aff_sum, epoch_ova_sum).detach().cpu())\n                if has_shared else np.nan\n            )\n\n            gates = {\n                "aff_private_gate": np.nan,\n                "ova_private_gate": np.nan,\n                "capacity_gate_1": np.nan,\n                "capacity_gate_2": np.nan,\n            }\n            if hasattr(self.model, "private_gate_values"):\n                gates.update(self.model.private_gate_values())\n            if hasattr(self.model, "capacity_gate_values"):\n                gates.update(self.model.capacity_gate_values())\n\n            row = {\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "arm": self.spec.name,\n                "arm_label": self.spec.label,\n                "architecture": self.spec.architecture,\n                "dominance_enabled": bool(self.spec.dominance),\n                "pcgrad_enabled": bool(self.spec.pcgrad),\n                "epoch": epoch + 1,\n                "dominance_active": dom_active,\n                "mean_total_loss": float(np.mean(totals)),\n                "mean_dominance_loss": float(np.mean(dom_values)),\n                "batch_conflict_rate": (\n                    float(np.mean(conflicts)) if conflicts else np.nan\n                ),\n                "mean_batch_pre_cosine": (\n                    float(np.mean(pre_cos)) if pre_cos else np.nan\n                ),\n                "mean_batch_post_cosine": (\n                    float(np.mean(post_cos)) if post_cos else np.nan\n                ),\n                "epoch_aggregate_cosine": epoch_global_cos,\n                "mean_aff_shared_grad_norm": (\n                    float(np.mean(aff_norms)) if aff_norms else np.nan\n                ),\n                "mean_ova_shared_grad_norm": (\n                    float(np.mean(ova_norms)) if ova_norms else np.nan\n                ),\n                "mean_projection_fraction": (\n                    float(np.mean(proj_fracs)) if proj_fracs else np.nan\n                ),\n                "mean_aff_private_to_shared_ratio": (\n                    float(np.mean(private_aff_ratios))\n                    if private_aff_ratios else np.nan\n                ),\n                "mean_ova_private_to_shared_ratio": (\n                    float(np.mean(private_ova_ratios))\n                    if private_ova_ratios else np.nan\n                ),\n                "mean_capacity_correction_to_shared_ratio": (\n                    float(np.mean(capacity_ratios))\n                    if capacity_ratios else np.nan\n                ),\n                **gates,\n            }\n            self.rows.append(row)\n\n            if epoch == 0 or (epoch + 1) % 5 == 0:\n                print(\n                    f"[{FEATURE_TYPE}] {self.spec.name} "\n                    f"epoch={epoch+1:02d}/{config.EPOCHS} "\n                    f"loss={row[\'mean_total_loss\']:.4f} "\n                    f"conflict={row[\'batch_conflict_rate\']}",\n                    flush=True,\n                )\n\n        return self\n\n\ndef class_pos_weight(y):\n    y = np.asarray(y, dtype=int)\n    positives = int((y == 1).sum())\n    negatives = int((y == 0).sum())\n    return float(negatives / max(positives, 1))\n\n\ndef fit_arm(X, y_aff, y_ova, arm_name):\n    spec = ARM_SPECS[arm_name]\n\n    # Reset to the SAME seed before each arm. Shared-base initial weights and\n    # minibatch order are therefore matched wherever architectures overlap.\n    hard_reset_rng(SEED, f"{FEATURE_TYPE} {arm_name}")\n\n    model = make_model(X.shape[1], spec)\n    trainer = ControlledTrainer(\n        model,\n        spec,\n        class_pos_weight(y_aff),\n        class_pos_weight(y_ova),\n    )\n    trainer.fit(\n        X,\n        y_aff,\n        y_ova,\n        f"{FEATURE_TYPE} seed={SEED} {arm_name}",\n    )\n    return model, trainer\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef arm_manifest_path(parent, arm):\n    return parent / arm / "manifest.json"\n\n\ndef manifest_matches(path, expected):\n    if not path.exists():\n        return False\n    try:\n        observed = json.loads(path.read_text(encoding="utf-8"))\n    except Exception:\n        return False\n    return all(observed.get(k) == v for k, v in expected.items())\n\n\ndef write_arm_outputs(\n    arm_dir,\n    manifest,\n    raw_df,\n    metrics_df,\n    diagnostics_df,\n    parameter_count,\n):\n    arm_dir.mkdir(parents=True, exist_ok=True)\n    raw_df.to_csv(arm_dir / "raw_predictions.csv.gz", index=False)\n    metrics_df.to_csv(arm_dir / "metrics.csv", index=False)\n    diagnostics_df.to_csv(\n        arm_dir / "training_diagnostics.csv.gz",\n        index=False,\n    )\n    pd.DataFrame([{\n        **manifest,\n        "trainable_parameters": int(parameter_count),\n    }]).to_csv(\n        arm_dir / "parameter_count.csv",\n        index=False,\n    )\n    (arm_dir / "manifest.json").write_text(\n        json.dumps(manifest, indent=2, sort_keys=True),\n        encoding="utf-8",\n    )\n\n\ndef run_smoke():\n    bundle = np.load(EMI_FEATURE_PATH, allow_pickle=False)\n    X = np.asarray(bundle[FEATURE_TYPE], dtype=np.float32)\n    y_aff = np.asarray(bundle["y_aff"], dtype=np.float32)\n    y_ova = np.asarray(bundle["y_ova"], dtype=np.float32)\n\n    selected = []\n    for a in [0, 1]:\n        for o in [0, 1]:\n            idx = np.flatnonzero(\n                (y_aff.astype(int) == a)\n                & (y_ova.astype(int) == o)\n            )\n            selected.extend(idx[:24].tolist())\n    selected = np.asarray(sorted(set(selected)), dtype=int)\n    if len(selected) < 32:\n        raise RuntimeError("Smoke subset is too small.")\n\n    old_epochs = config.EPOCHS\n    config.EPOCHS = 1\n    try:\n        rows = []\n        for arm in CORE_ARMS:\n            model, trainer = fit_arm(\n                X[selected],\n                y_aff[selected],\n                y_ova[selected],\n                arm,\n            )\n            pred_aff, pred_ova = predict_torch_model(\n                model, X[selected[:8]]\n            )\n            if not (\n                np.isfinite(pred_aff).all()\n                and np.isfinite(pred_ova).all()\n            ):\n                raise RuntimeError(f"Nonfinite smoke prediction: {arm}")\n            rows.append({\n                "arm": arm,\n                "parameter_count": trainable_parameter_count(model),\n                "diagnostic_rows": len(trainer.rows),\n            })\n            del model, trainer\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n            gc.collect()\n        pd.DataFrame(rows).to_csv(\n            OUT_DIR / "smoke_summary.csv",\n            index=False,\n        )\n        print("COMPONENT ABLATION RUNTIME PREFLIGHT PASS")\n    finally:\n        config.EPOCHS = old_epochs\n\n\ndef run_mutation():\n    if HOLDOUT_PATH is None:\n        raise RuntimeError("Mutation stage requires MOLM_HOLDOUT_PATH.")\n\n    bundle = np.load(EMI_FEATURE_PATH, allow_pickle=False)\n    X = np.asarray(bundle[FEATURE_TYPE], dtype=np.float32)\n    y_aff = np.asarray(bundle["y_aff"], dtype=np.float32)\n    y_ova = np.asarray(bundle["y_ova"], dtype=np.float32)\n    sequences = bundle["sequences"].astype(str)\n    holdouts = json.loads(HOLDOUT_PATH.read_text(encoding="utf-8"))\n\n    for holdout in holdouts:\n        holdout_id = holdout["holdout_id"]\n        train_idx = np.asarray(holdout["train_idx"], dtype=int)\n        test_idx = np.asarray(holdout["test_idx"], dtype=int)\n        holdout_dir = OUT_DIR / holdout_id\n\n        for arm in ARMS:\n            expected = {\n                "lock_sha256": LOCK_HASH,\n                "stage": "mutation",\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "arm": arm,\n                "holdout_id": holdout_id,\n            }\n            manifest_path = arm_manifest_path(holdout_dir, arm)\n            if manifest_matches(manifest_path, expected):\n                print(\n                    f"SKIP VERIFIED seed={SEED} feature={FEATURE_TYPE} "\n                    f"holdout={holdout_id} arm={arm}",\n                    flush=True,\n                )\n                continue\n\n            print(\n                f"TRAIN seed={SEED} feature={FEATURE_TYPE} "\n                f"holdout={holdout_id} arm={arm} "\n                f"train={len(train_idx)} test={len(test_idx)}",\n                flush=True,\n            )\n            model, trainer = fit_arm(\n                X[train_idx],\n                y_aff[train_idx],\n                y_ova[train_idx],\n                arm,\n            )\n            pred_aff, pred_ova = predict_torch_model(\n                model, X[test_idx]\n            )\n\n            raw = pd.DataFrame({\n                "stage": "mutation",\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "arm": arm,\n                "arm_label": ARM_SPECS[arm].label,\n                "holdout_id": holdout_id,\n                "kabat_site": int(holdout["kabat_site"]),\n                "python_index": int(holdout["python_index"]),\n                "heldout_residue": holdout["heldout_residue"],\n                "row_id": test_idx,\n                "sequence_id": sequences[test_idx],\n                "true_aff": y_aff[test_idx],\n                "true_ova": y_ova[test_idx],\n                "pred_aff": pred_aff,\n                "pred_ova": pred_ova,\n            })\n\n            metric_rows = []\n            for task, truth, score in [\n                ("affinity", y_aff[test_idx], pred_aff),\n                ("ova", y_ova[test_idx], pred_ova),\n            ]:\n                metric_rows.append({\n                    "stage": "mutation",\n                    "seed": SEED,\n                    "feature": FEATURE_TYPE,\n                    "arm": arm,\n                    "arm_label": ARM_SPECS[arm].label,\n                    "holdout_id": holdout_id,\n                    "kabat_site": int(holdout["kabat_site"]),\n                    "heldout_residue": holdout["heldout_residue"],\n                    "task": task,\n                    **binary_metrics(truth, score),\n                })\n\n            write_arm_outputs(\n                holdout_dir / arm,\n                expected,\n                raw,\n                pd.DataFrame(metric_rows),\n                pd.DataFrame(trainer.rows),\n                trainable_parameter_count(model),\n            )\n\n            del model, trainer\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n            gc.collect()\n\n\ndef run_external():\n    if EXTERNAL_FEATURE_PATH is None:\n        raise RuntimeError("External stage requires MOLM_EXTERNAL_FEATURE_PATH.")\n\n    emi = np.load(EMI_FEATURE_PATH, allow_pickle=False)\n    ext = np.load(EXTERNAL_FEATURE_PATH, allow_pickle=False)\n\n    X_train = np.asarray(emi[FEATURE_TYPE], dtype=np.float32)\n    y_aff_train = np.asarray(emi["y_aff"], dtype=np.float32)\n    y_ova_train = np.asarray(emi["y_ova"], dtype=np.float32)\n\n    dataset_specs = [\n        ("ISO", "iso", np.arange(int(ext["iso_n"]), dtype=int)),\n        (\n            "IgG-primary42",\n            "igg",\n            np.asarray(ext["igg_primary42_indices"], dtype=int),\n        ),\n        (\n            "IgG-all96",\n            "igg",\n            np.arange(int(ext["igg_n"]), dtype=int),\n        ),\n    ]\n\n    for arm in ARMS:\n        arm_dir = OUT_DIR / arm\n        expected = {\n            "lock_sha256": LOCK_HASH,\n            "stage": "external",\n            "seed": SEED,\n            "feature": FEATURE_TYPE,\n            "arm": arm,\n        }\n        manifest_path = arm_manifest_path(OUT_DIR, arm)\n        if manifest_matches(manifest_path, expected):\n            print(\n                f"SKIP VERIFIED external seed={SEED} "\n                f"feature={FEATURE_TYPE} arm={arm}",\n                flush=True,\n            )\n            continue\n\n        model, trainer = fit_arm(\n            X_train,\n            y_aff_train,\n            y_ova_train,\n            arm,\n        )\n\n        raw_frames = []\n        metric_rows = []\n\n        for dataset_name, prefix, indices in dataset_specs:\n            X_eval = np.asarray(\n                ext[f"{prefix}__{FEATURE_TYPE}"][indices],\n                dtype=np.float32,\n            )\n            true_aff = np.asarray(\n                ext[f"{prefix}__y_aff"][indices],\n                dtype=float,\n            )\n            true_ova = np.asarray(\n                ext[f"{prefix}__y_ova"][indices],\n                dtype=float,\n            )\n            seqs = ext[f"{prefix}__sequences"][indices].astype(str)\n            source_rows = np.asarray(indices, dtype=int)\n            sample_ids = (\n                ext["igg__sample_ids"][indices].astype(str)\n                if prefix == "igg"\n                else np.asarray([""] * len(indices), dtype=str)\n            )\n\n            pred_aff, pred_ova = predict_torch_model(\n                model, X_eval\n            )\n\n            raw_frames.append(pd.DataFrame({\n                "stage": "external",\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "arm": arm,\n                "arm_label": ARM_SPECS[arm].label,\n                "dataset": dataset_name,\n                "row_id": np.arange(len(indices), dtype=int),\n                "source_row_id": source_rows,\n                "sequence_id": seqs,\n                "sample_id": sample_ids,\n                "true_aff": true_aff,\n                "true_ova": true_ova,\n                "pred_aff": pred_aff,\n                "pred_ova": pred_ova,\n            }))\n\n            metric_rows.append({\n                "stage": "external",\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "arm": arm,\n                "arm_label": ARM_SPECS[arm].label,\n                "dataset": dataset_name,\n                "aff_spearman": float(\n                    spearmanr(true_aff, pred_aff).statistic\n                ),\n                "ova_spearman": float(\n                    spearmanr(true_ova, pred_ova).statistic\n                ),\n                "n": int(len(indices)),\n            })\n\n        write_arm_outputs(\n            arm_dir,\n            expected,\n            pd.concat(raw_frames, ignore_index=True),\n            pd.DataFrame(metric_rows),\n            pd.DataFrame(trainer.rows),\n            trainable_parameter_count(model),\n        )\n\n        del model, trainer\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n        gc.collect()\n\n\nif __name__ == "__main__":\n    start = time.time()\n    print(\n        f"stage={STAGE} seed={SEED} feature={FEATURE_TYPE} "\n        f"arm_set={ARM_SET} device="\n        f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'cpu\'}",\n        flush=True,\n    )\n\n    if SMOKE or STAGE == "smoke":\n        run_smoke()\n    elif STAGE == "mutation":\n        run_mutation()\n    elif STAGE == "external":\n        run_external()\n    else:\n        raise ValueError(f"Unknown stage={STAGE!r}")\n\n    print(f"DONE in {(time.time() - start)/60:.2f} min", flush=True)\n'
LOSS_WORKER_SOURCE = 'from __future__ import annotations\n\nimport gc\nimport hashlib\nimport json\nimport os\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom scipy.stats import spearmanr\n\nREPO = Path(os.environ["MOLM_REPO"]).resolve()\nFEATURE_TYPE = os.environ.get("MOLM_FEATURE_TYPE", "onehot")\nSEED = int(os.environ.get("MOLM_SEED", "42"))\nOUT_DIR = Path(os.environ["MOLM_JOB_OUTPUT"]).resolve()\nEMI_FEATURE_PATH = Path(os.environ["MOLM_EMI_FEATURE_PATH"]).resolve()\nEXTERNAL_FEATURE_PATH = Path(os.environ["MOLM_EXTERNAL_FEATURE_PATH"]).resolve()\nLOCK_HASH = os.environ["MOLM_LOCK_HASH"]\nSMOKE = os.environ.get("MOLM_SMOKE", "0") == "1"\n\nOUT_DIR.mkdir(parents=True, exist_ok=True)\n\nimport sys\nsys.path.insert(0, str(REPO))\nfrom phase0_config import (  # noqa: E402\n    DiagnosticMOLM,\n    config,\n    focal_bce_with_logits,\n    gap_hinge_loss,\n    hard_reset_rng,\n    ranking_loss,\n)\n\nconfig.PARETO_LOSS = False\nconfig.EPOCHS = int(os.environ.get("MOLM_EPOCHS", "25"))\nconfig.BATCH_SIZE = int(os.environ.get("MOLM_BATCH_SIZE", "64"))\nconfig.LEARNING_RATE = float(os.environ.get("MOLM_LEARNING_RATE", "5e-5"))\n\nEXPECTED = {\n    "RANKING_WEIGHT_AFF": 0.3,\n    "RANKING_WEIGHT_SPEC": 0.6,\n    "GAP_WEIGHT_AFF": 0.2,\n    "GAP_WEIGHT_SPEC": 0.6,\n    "RANKING_MARGIN": 0.3,\n    "GAP_MARGIN": 0.2,\n    "FOCAL_GAMMA": 2.0,\n}\nfor key, expected in EXPECTED.items():\n    observed = float(getattr(config, key))\n    if not np.isclose(observed, expected):\n        raise RuntimeError(\n            f"Pinned loss setting changed: {key}={observed}, expected={expected}"\n        )\n\n# Pre-specified contrasts for the loss and architecture component analyses.\nARMS = {\n    "S0_shared_focal": {\n        "label": "S0 Shared + Focal",\n        "architecture": "shared",\n        "ranking": False,\n        "gap": False,\n    },\n    "SR_shared_focal_ranking": {\n        "label": "SR Shared + Focal + Ranking",\n        "architecture": "shared",\n        "ranking": True,\n        "gap": False,\n    },\n    "SG_shared_focal_gap": {\n        "label": "SG Shared + Focal + Gap",\n        "architecture": "shared",\n        "ranking": False,\n        "gap": True,\n    },\n    "SRG_shared_full": {\n        "label": "SRG Shared + Focal + Ranking + Gap",\n        "architecture": "shared",\n        "ranking": True,\n        "gap": True,\n    },\n    "IRG_independent_full": {\n        "label": "IRG Independent + Focal + Ranking + Gap",\n        "architecture": "independent",\n        "ranking": True,\n        "gap": True,\n    },\n}\nARM_ORDER = list(ARMS)\n\n\ndef class_pos_weight(y):\n    y = np.asarray(y, dtype=int)\n    positives = int((y == 1).sum())\n    negatives = int((y == 0).sum())\n    return float(negatives / max(positives, 1))\n\n\ndef trainable_parameter_count(model):\n    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))\n\n\ndef model_parameter_sha256(model):\n    h = hashlib.sha256()\n    for name, tensor in model.state_dict().items():\n        h.update(name.encode("utf-8"))\n        h.update(np.asarray(tensor.detach().cpu()).tobytes())\n    return h.hexdigest()\n\n\nclass IndependentTaskNet(nn.Module):\n    """One task network matched to shared-encoder + task-tower depth."""\n    def __init__(self, input_dim):\n        super().__init__()\n        dims = list(config.SHARED_DIMS) + list(config.TOWER_DIMS)\n        self.blocks = DiagnosticMOLM._make_blocks(\n            input_dim, dims, config.DROPOUT_RATE\n        )\n        output_dim = dims[-1] if dims else input_dim\n        self.proj = nn.Linear(output_dim, config.LATENT_DIM)\n        self.proj_norm = nn.LayerNorm(config.LATENT_DIM)\n        self.head = nn.Linear(config.LATENT_DIM, 1)\n\n    @staticmethod\n    def _dropout(layer, inputs, training):\n        if training is None:\n            return layer(inputs)\n        return F.dropout(inputs, p=layer.p, training=training)\n\n    def forward(self, inputs, training=None):\n        for index in range(0, len(self.blocks), 3):\n            inputs = self.blocks[index](inputs)\n            inputs = self.blocks[index + 1](inputs)\n            inputs = F.gelu(inputs)\n            inputs = self._dropout(self.blocks[index + 2], inputs, training)\n        latent = self.proj_norm(self.proj(inputs))\n        return self.head(latent).squeeze(-1)\n\n\nclass IndependentNNPair(nn.Module):\n    """Two independently parameterized task networks."""\n    def __init__(self, input_dim):\n        super().__init__()\n        self.aff_net = IndependentTaskNet(input_dim)\n        self.ova_net = IndependentTaskNet(input_dim)\n\n    def forward(self, inputs, training=None):\n        return {\n            "aff_score": self.aff_net(inputs, training=training),\n            "spec_score": self.ova_net(inputs, training=training),\n        }\n\n\ndef make_model(input_dim, architecture):\n    if architecture == "shared":\n        return DiagnosticMOLM(\n            input_dim=input_dim,\n            latent_dim=config.LATENT_DIM,\n            shared_dims=config.SHARED_DIMS,\n            tower_dims=config.TOWER_DIMS,\n            dropout_rate=config.DROPOUT_RATE,\n            grl_lambda=config.GRL_LAMBDA,\n        )\n    if architecture == "independent":\n        return IndependentNNPair(input_dim)\n    raise ValueError(architecture)\n\n\ndef loss_components(outputs, y_aff, y_ova, aff_pw, ova_pw, arm_name):\n    spec = ARMS[arm_name]\n    aff_focal = focal_bce_with_logits(\n        y_aff, outputs["aff_score"], config.FOCAL_GAMMA, aff_pw\n    )\n    ova_focal = focal_bce_with_logits(\n        y_ova, outputs["spec_score"], config.FOCAL_GAMMA, ova_pw\n    )\n    aff_rank_raw = ranking_loss(\n        outputs["aff_score"], y_aff, config.RANKING_MARGIN\n    )\n    ova_rank_raw = ranking_loss(\n        outputs["spec_score"], y_ova, config.RANKING_MARGIN\n    )\n    aff_gap_raw = gap_hinge_loss(\n        outputs["aff_score"], y_aff, config.GAP_MARGIN\n    )\n    ova_gap_raw = gap_hinge_loss(\n        outputs["spec_score"], y_ova, config.GAP_MARGIN\n    )\n    zero_aff = outputs["aff_score"].sum() * 0.0\n    zero_ova = outputs["spec_score"].sum() * 0.0\n    aff_rank = (\n        config.RANKING_WEIGHT_AFF * aff_rank_raw\n        if spec["ranking"] else zero_aff\n    )\n    ova_rank = (\n        config.RANKING_WEIGHT_SPEC * ova_rank_raw\n        if spec["ranking"] else zero_ova\n    )\n    aff_gap = (\n        config.GAP_WEIGHT_AFF * aff_gap_raw\n        if spec["gap"] else zero_aff\n    )\n    ova_gap = (\n        config.GAP_WEIGHT_SPEC * ova_gap_raw\n        if spec["gap"] else zero_ova\n    )\n    aff_total = aff_focal + aff_rank + aff_gap\n    ova_total = ova_focal + ova_rank + ova_gap\n    return {\n        "total": aff_total + ova_total,\n        "aff_total": aff_total,\n        "ova_total": ova_total,\n        "aff_focal": aff_focal,\n        "ova_focal": ova_focal,\n        "aff_rank_weighted": aff_rank,\n        "ova_rank_weighted": ova_rank,\n        "aff_gap_weighted": aff_gap,\n        "ova_gap_weighted": ova_gap,\n    }\n\n\nclass Trainer:\n    def __init__(self, model, arm_name, aff_pw, ova_pw, initial_hash):\n        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n        self.model = model.to(self.device)\n        self.arm_name = arm_name\n        self.aff_pw = float(aff_pw)\n        self.ova_pw = float(ova_pw)\n        self.initial_hash = initial_hash\n        self.optimizer = torch.optim.AdamW(\n            self.model.parameters(), lr=config.LEARNING_RATE, weight_decay=1e-4\n        )\n        self.rows = []\n\n    def fit(self, X, y_aff, y_ova):\n        dataset = torch.utils.data.TensorDataset(\n            torch.as_tensor(X, dtype=torch.float32),\n            torch.as_tensor(y_aff, dtype=torch.float32),\n            torch.as_tensor(y_ova, dtype=torch.float32),\n        )\n        # Same minibatch order for all arms within a seed.\n        generator = torch.Generator().manual_seed(SEED)\n        loader = torch.utils.data.DataLoader(\n            dataset,\n            batch_size=config.BATCH_SIZE,\n            shuffle=True,\n            generator=generator,\n        )\n        for epoch in range(config.EPOCHS):\n            acc = {\n                "total": [], "aff_total": [], "ova_total": [],\n                "aff_focal": [], "ova_focal": [],\n                "aff_rank_weighted": [], "ova_rank_weighted": [],\n                "aff_gap_weighted": [], "ova_gap_weighted": [],\n            }\n            for Xb, yab, yob in loader:\n                Xb = Xb.to(self.device)\n                yab = yab.to(self.device)\n                yob = yob.to(self.device)\n                self.model.train()\n                self.optimizer.zero_grad(set_to_none=True)\n                out = self.model(Xb, training=True)\n                comp = loss_components(\n                    out, yab, yob, self.aff_pw, self.ova_pw, self.arm_name\n                )\n                comp["total"].backward()\n                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)\n                self.optimizer.step()\n                for key in acc:\n                    acc[key].append(float(comp[key].detach().cpu()))\n            row = {\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "arm": self.arm_name,\n                "arm_label": ARMS[self.arm_name]["label"],\n                "epoch": epoch + 1,\n            }\n            row.update({f"mean_{k}": float(np.mean(v)) for k, v in acc.items()})\n            self.rows.append(row)\n\n\ndef fit_arm(X, y_aff, y_ova, arm_name):\n    spec = ARMS[arm_name]\n    hard_reset_rng(SEED, f"pareto_component {FEATURE_TYPE} {arm_name}")\n    model = make_model(X.shape[1], spec["architecture"])\n    initial_hash = model_parameter_sha256(model)\n    trainer = Trainer(\n        model,\n        arm_name,\n        class_pos_weight(y_aff),\n        class_pos_weight(y_ova),\n        initial_hash,\n    )\n    trainer.fit(X, y_aff, y_ova)\n    return model, trainer\n\n\ndef predict_model(model, X):\n    model.eval()\n    device = next(model.parameters()).device\n    with torch.no_grad():\n        out = model(torch.as_tensor(X, dtype=torch.float32, device=device), training=False)\n    return (\n        out["aff_score"].detach().cpu().numpy().reshape(-1),\n        out["spec_score"].detach().cpu().numpy().reshape(-1),\n    )\n\n\ndef manifest_matches(path, expected):\n    if not path.exists():\n        return False\n    try:\n        observed = json.loads(path.read_text(encoding="utf-8"))\n    except Exception:\n        return False\n    return all(observed.get(k) == v for k, v in expected.items())\n\n\ndef run_external():\n    emi = np.load(EMI_FEATURE_PATH, allow_pickle=False)\n    ext = np.load(EXTERNAL_FEATURE_PATH, allow_pickle=False)\n    X_train = np.asarray(emi[FEATURE_TYPE], dtype=np.float32)\n    y_aff_train = np.asarray(emi["y_aff"], dtype=np.float32)\n    y_ova_train = np.asarray(emi["y_ova"], dtype=np.float32)\n\n    if SMOKE:\n        keep = np.arange(min(512, len(X_train)), dtype=int)\n        X_train = X_train[keep]\n        y_aff_train = y_aff_train[keep]\n        y_ova_train = y_ova_train[keep]\n        config.EPOCHS = 1\n\n    dataset_specs = [\n        ("ISO", "iso", np.arange(int(ext["iso_n"]), dtype=int)),\n        ("IgG-primary42", "igg", np.asarray(ext["igg_primary42_indices"], dtype=int)),\n        ("IgG-all96", "igg", np.arange(int(ext["igg_n"]), dtype=int)),\n    ]\n\n    shared_hashes = []\n    audit_rows = []\n    for arm in ARM_ORDER:\n        spec = ARMS[arm]\n        arm_dir = OUT_DIR / arm\n        expected = {\n            "lock_sha256": LOCK_HASH,\n            "seed": SEED,\n            "feature": FEATURE_TYPE,\n            "arm": arm,\n            "stage": "external_pareto_component_ablation",\n        }\n        manifest_path = arm_dir / "manifest.json"\n        if (not SMOKE) and manifest_matches(manifest_path, expected):\n            print(f"SKIP VERIFIED seed={SEED} feature={FEATURE_TYPE} arm={arm}", flush=True)\n            continue\n\n        print(f"TRAIN seed={SEED} feature={FEATURE_TYPE} arm={arm}", flush=True)\n        model, trainer = fit_arm(X_train, y_aff_train, y_ova_train, arm)\n        if spec["architecture"] == "shared":\n            shared_hashes.append(trainer.initial_hash)\n\n        raw_frames = []\n        metric_rows = []\n        for dataset_name, prefix, indices in dataset_specs:\n            X_eval = np.asarray(ext[f"{prefix}__{FEATURE_TYPE}"][indices], dtype=np.float32)\n            true_aff = np.asarray(ext[f"{prefix}__y_aff"][indices], dtype=float)\n            true_ova = np.asarray(ext[f"{prefix}__y_ova"][indices], dtype=float)\n            seqs = ext[f"{prefix}__sequences"][indices].astype(str)\n            sample_ids = (\n                ext["igg__sample_ids"][indices].astype(str)\n                if prefix == "igg"\n                else np.asarray([""] * len(indices), dtype=str)\n            )\n            pred_aff, pred_ova = predict_model(model, X_eval)\n            if not (np.isfinite(pred_aff).all() and np.isfinite(pred_ova).all()):\n                raise RuntimeError(f"Nonfinite predictions for {arm} {dataset_name}")\n            raw_frames.append(pd.DataFrame({\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "arm": arm,\n                "arm_label": spec["label"],\n                "dataset": dataset_name,\n                "row_id": np.arange(len(indices), dtype=int),\n                "source_row_id": np.asarray(indices, dtype=int),\n                "sequence_id": seqs,\n                "sample_id": sample_ids,\n                "true_aff": true_aff,\n                "true_ova": true_ova,\n                "pred_aff": pred_aff,\n                "pred_ova": pred_ova,\n            }))\n            metric_rows.append({\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "arm": arm,\n                "arm_label": spec["label"],\n                "dataset": dataset_name,\n                "n": int(len(indices)),\n                "aff_spearman": float(spearmanr(true_aff, pred_aff).statistic),\n                "ova_spearman": float(spearmanr(true_ova, pred_ova).statistic),\n            })\n\n        arm_dir.mkdir(parents=True, exist_ok=True)\n        pd.concat(raw_frames, ignore_index=True).to_csv(\n            arm_dir / "raw_predictions.csv.gz", index=False, compression="gzip"\n        )\n        pd.DataFrame(metric_rows).to_csv(arm_dir / "external_spearman.csv", index=False)\n        pd.DataFrame(trainer.rows).to_csv(\n            arm_dir / "training_diagnostics.csv.gz", index=False, compression="gzip"\n        )\n        audit = {\n            **expected,\n            "architecture": spec["architecture"],\n            "ranking_enabled": bool(spec["ranking"]),\n            "gap_enabled": bool(spec["gap"]),\n            "trainable_parameters": trainable_parameter_count(model),\n            "initial_param_sha256": trainer.initial_hash,\n        }\n        audit_rows.append(audit)\n        pd.DataFrame([audit]).to_csv(arm_dir / "model_audit.csv", index=False)\n        manifest_path.write_text(json.dumps(expected, indent=2, sort_keys=True), encoding="utf-8")\n\n        del model, trainer\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n        gc.collect()\n\n    # Exact fairness invariant for the four shared-loss arms.\n    if shared_hashes and len(set(shared_hashes)) != 1:\n        raise RuntimeError("Shared loss arms did not start from identical parameters.")\n    if SMOKE:\n        pd.DataFrame(audit_rows).to_csv(OUT_DIR / "smoke_audit.csv", index=False)\n        print("PARETO COMPONENT ABLATION SMOKE PASS", flush=True)\n\n\nif __name__ == "__main__":\n    start = time.time()\n    print(\n        f"seed={SEED} feature={FEATURE_TYPE} "\n        f"device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'cpu\'}",\n        flush=True,\n    )\n    run_external()\n    print(f"DONE in {(time.time()-start)/60:.2f} min", flush=True)\n'

COMMON_PATH = CODE_DIR/'molm_definitive_common.py'
ARCH_WORKER_PATH = CODE_DIR/'molm_pareto_architecture_8arm_worker.py'
LOSS_WORKER_PATH = CODE_DIR/'molm_pareto_loss_ablation_worker.py'

COMMON_PATH.write_text(COMMON_SOURCE, encoding='utf-8')
ARCH_WORKER_PATH.write_text(ARCH_WORKER_SOURCE, encoding='utf-8')
LOSS_WORKER_PATH.write_text(LOSS_WORKER_SOURCE, encoding='utf-8')

compile(COMMON_SOURCE, str(COMMON_PATH), 'exec')
compile(ARCH_WORKER_SOURCE, str(ARCH_WORKER_PATH), 'exec')
compile(LOSS_WORKER_SOURCE, str(LOSS_WORKER_PATH), 'exec')

print('Common helper:', COMMON_PATH)
print('Architecture worker:', ARCH_WORKER_PATH)
print('Loss worker:', LOSS_WORKER_PATH)
print('Architecture worker SHA256:', hashlib.sha256(ARCH_WORKER_SOURCE.encode()).hexdigest())
print('Loss worker SHA256:', hashlib.sha256(LOSS_WORKER_SOURCE.encode()).hexdigest())


## 5. Smoke tests
The architecture smoke test verifies that **all eight A–H arms**, including Full Routed-MOLM, instantiate and train. The loss smoke test verifies all five loss/share controls and checks identical initialization across the four shared loss arms.


In [ ]:
ARCH_SMOKE_DIR=WORK_ROOT/'smoke_architecture'
LOSS_SMOKE_DIR=WORK_ROOT/'smoke_loss'
ARCH_SMOKE_DIR.mkdir(exist_ok=True)
LOSS_SMOKE_DIR.mkdir(exist_ok=True)

# A-H architecture smoke test.
arch_smoke_log=LOG_DIR/'smoke_architecture.log'
env=os.environ.copy()
env.update({
    'CUDA_VISIBLE_DEVICES':'0',
    'MOLM_REPO':str(REPO),
    'MOLM_FEATURE_TYPE':'onehot',
    'MOLM_SEED':str(SEEDS[0]),
    'MOLM_STAGE':'smoke',
    'MOLM_ARM_SET':'core',
    'MOLM_JOB_OUTPUT':str(ARCH_SMOKE_DIR),
    'MOLM_EMI_FEATURE_PATH':str(EMI_FEATURE_PATH),
    'MOLM_EXTERNAL_FEATURE_PATH':str(EXTERNAL_FEATURE_PATH),
    'MOLM_LOCK_HASH':LOCK_SHA,
    'MOLM_EPOCHS':'1',
    'MOLM_BATCH_SIZE':str(BATCH_SIZE),
    'MOLM_LEARNING_RATE':str(LEARNING_RATE),
    'MOLM_SMOKE':'1',
    'MOLM_DOMINANCE_WEIGHT':str(ARCH_COMPONENT_SETTINGS['DOMINANCE_WEIGHT']),
    'MOLM_DOMINANCE_MARGIN':str(ARCH_COMPONENT_SETTINGS['DOMINANCE_MARGIN']),
    'MOLM_DOMINANCE_WARMUP':str(ARCH_COMPONENT_SETTINGS['DOMINANCE_WARMUP']),
    'MOLM_DOMINANCE_MAX_PAIRS':str(ARCH_COMPONENT_SETTINGS['DOMINANCE_MAX_PAIRS']),
    'MOLM_PRIVATE_ADAPTER_DIM':str(ARCH_COMPONENT_SETTINGS['PRIVATE_ADAPTER_DIM']),
    'MOLM_PRIVATE_GATE_INIT':str(ARCH_COMPONENT_SETTINGS['PRIVATE_GATE_INIT']),
    'MOLM_ROUTING_EPS':str(ARCH_COMPONENT_SETTINGS['ROUTING_EPS']),
})
with arch_smoke_log.open('w') as f:
    p=subprocess.run([sys.executable,'-u',str(ARCH_WORKER_PATH)],env=env,stdout=f,stderr=subprocess.STDOUT,text=True)
if p.returncode!=0:
    raise RuntimeError('ARCHITECTURE SMOKE FAILED\n'+'\n'.join(arch_smoke_log.read_text(errors='replace').splitlines()[-160:]))
print('\n'.join(arch_smoke_log.read_text(errors='replace').splitlines()[-20:]))

# Focal/ranking/gap smoke test.
loss_smoke_log=LOG_DIR/'smoke_loss.log'
env=os.environ.copy()
env.update({
    'CUDA_VISIBLE_DEVICES':'0',
    'MOLM_REPO':str(REPO),
    'MOLM_FEATURE_TYPE':'onehot',
    'MOLM_SEED':str(SEEDS[0]),
    'MOLM_JOB_OUTPUT':str(LOSS_SMOKE_DIR),
    'MOLM_EMI_FEATURE_PATH':str(EMI_FEATURE_PATH),
    'MOLM_EXTERNAL_FEATURE_PATH':str(EXTERNAL_FEATURE_PATH),
    'MOLM_LOCK_HASH':LOCK_SHA,
    'MOLM_EPOCHS':'1',
    'MOLM_BATCH_SIZE':str(BATCH_SIZE),
    'MOLM_LEARNING_RATE':str(LEARNING_RATE),
    'MOLM_SMOKE':'1',
})
with loss_smoke_log.open('w') as f:
    p=subprocess.run([sys.executable,'-u',str(LOSS_WORKER_PATH)],env=env,stdout=f,stderr=subprocess.STDOUT,text=True)
if p.returncode!=0:
    raise RuntimeError('LOSS SMOKE FAILED\n'+'\n'.join(loss_smoke_log.read_text(errors='replace').splitlines()[-160:]))
print('\n'.join(loss_smoke_log.read_text(errors='replace').splitlines()[-20:]))
print('BOTH SMOKE TESTS PASS')


## 6. Full five-seed run on T4 ×2
The notebook now runs **5 seeds × (8 architecture arms + 5 loss/share arms) = 65 fits** for the default OneHot experiment.

Each seed job runs the A–H architecture suite and then the loss suite on the same GPU. Five seed jobs are distributed across the two GPUs. Resume logic verifies manifests independently for both suites.


In [ ]:
def arch_job_complete(root, seed, feature):
    for arm in ARCH_ARMS:
        manifest=root/arm/'manifest.json'
        if not manifest.exists():
            return False
        try:
            d=json.loads(manifest.read_text())
        except Exception:
            return False
        expected={'lock_sha256':LOCK_SHA,'stage':'external','seed':seed,'feature':feature,'arm':arm}
        if not all(d.get(k)==v for k,v in expected.items()):
            return False
    return True

def loss_job_complete(root, seed, feature):
    for arm in LOSS_ARMS:
        manifest=root/arm/'manifest.json'
        if not manifest.exists():
            return False
        try:
            d=json.loads(manifest.read_text())
        except Exception:
            return False
        expected={
            'lock_sha256':LOCK_SHA,'seed':seed,'feature':feature,'arm':arm,
            'stage':'external_pareto_component_ablation',
        }
        if not all(d.get(k)==v for k,v in expected.items()):
            return False
    return True

def _run_worker(worker_path, env, log_path, label):
    with log_path.open('w') as f:
        p=subprocess.run([sys.executable,'-u',str(worker_path)],env=env,stdout=f,stderr=subprocess.STDOUT,text=True)
    if p.returncode!=0:
        raise RuntimeError(label+' FAILED\n'+'\n'.join(log_path.read_text(errors='replace').splitlines()[-160:]))

def run_seed_job(job,gpu):
    seed,feature=job

    # Architecture A-H.
    arch_root=ARCH_RUN_DIR/f'seed_{seed}'/feature
    arch_root.mkdir(parents=True,exist_ok=True)
    arch_status='skipped_verified'
    if not (RESUME and arch_job_complete(arch_root,seed,feature)):
        env=os.environ.copy()
        env.update({
            'CUDA_VISIBLE_DEVICES':str(gpu),
            'MOLM_REPO':str(REPO),
            'MOLM_FEATURE_TYPE':feature,
            'MOLM_SEED':str(seed),
            'MOLM_STAGE':'external',
            'MOLM_ARM_SET':'core',
            'MOLM_JOB_OUTPUT':str(arch_root),
            'MOLM_EMI_FEATURE_PATH':str(EMI_FEATURE_PATH),
            'MOLM_EXTERNAL_FEATURE_PATH':str(EXTERNAL_FEATURE_PATH),
            'MOLM_LOCK_HASH':LOCK_SHA,
            'MOLM_EPOCHS':str(EPOCHS),
            'MOLM_BATCH_SIZE':str(BATCH_SIZE),
            'MOLM_LEARNING_RATE':str(LEARNING_RATE),
            'MOLM_SMOKE':'0',
            'MOLM_DOMINANCE_WEIGHT':str(ARCH_COMPONENT_SETTINGS['DOMINANCE_WEIGHT']),
            'MOLM_DOMINANCE_MARGIN':str(ARCH_COMPONENT_SETTINGS['DOMINANCE_MARGIN']),
            'MOLM_DOMINANCE_WARMUP':str(ARCH_COMPONENT_SETTINGS['DOMINANCE_WARMUP']),
            'MOLM_DOMINANCE_MAX_PAIRS':str(ARCH_COMPONENT_SETTINGS['DOMINANCE_MAX_PAIRS']),
            'MOLM_PRIVATE_ADAPTER_DIM':str(ARCH_COMPONENT_SETTINGS['PRIVATE_ADAPTER_DIM']),
            'MOLM_PRIVATE_GATE_INIT':str(ARCH_COMPONENT_SETTINGS['PRIVATE_GATE_INIT']),
            'MOLM_ROUTING_EPS':str(ARCH_COMPONENT_SETTINGS['ROUTING_EPS']),
        })
        _run_worker(
            ARCH_WORKER_PATH, env,
            LOG_DIR/f'pareto_arch_seed{seed}_{feature}.log',
            f'architecture seed={seed} feature={feature}',
        )
        if not arch_job_complete(arch_root,seed,feature):
            raise RuntimeError(f'Architecture outputs incomplete: {arch_root}')
        arch_status='ok'

    # Loss/share controls.
    loss_root=LOSS_RUN_DIR/f'seed_{seed}'/feature
    loss_root.mkdir(parents=True,exist_ok=True)
    loss_status='skipped_verified'
    if not (RESUME and loss_job_complete(loss_root,seed,feature)):
        env=os.environ.copy()
        env.update({
            'CUDA_VISIBLE_DEVICES':str(gpu),
            'MOLM_REPO':str(REPO),
            'MOLM_FEATURE_TYPE':feature,
            'MOLM_SEED':str(seed),
            'MOLM_JOB_OUTPUT':str(loss_root),
            'MOLM_EMI_FEATURE_PATH':str(EMI_FEATURE_PATH),
            'MOLM_EXTERNAL_FEATURE_PATH':str(EXTERNAL_FEATURE_PATH),
            'MOLM_LOCK_HASH':LOCK_SHA,
            'MOLM_EPOCHS':str(EPOCHS),
            'MOLM_BATCH_SIZE':str(BATCH_SIZE),
            'MOLM_LEARNING_RATE':str(LEARNING_RATE),
            'MOLM_SMOKE':'0',
        })
        _run_worker(
            LOSS_WORKER_PATH, env,
            LOG_DIR/f'pareto_loss_seed{seed}_{feature}.log',
            f'loss seed={seed} feature={feature}',
        )
        if not loss_job_complete(loss_root,seed,feature):
            raise RuntimeError(f'Loss outputs incomplete: {loss_root}')
        loss_status='ok'

    return {
        'seed':seed,'feature':feature,'gpu':gpu,
        'architecture_status':arch_status,'loss_status':loss_status,
    }

def run_parallel_jobs(jobs, gpu_ids=(0,1)):
    q=queue.Queue()
    for j in jobs: q.put(j)
    rows=[]; lock=threading.Lock()
    def loop(gpu):
        while True:
            try: job=q.get_nowait()
            except queue.Empty: return
            try: rec=run_seed_job(job,gpu)
            except Exception as e:
                rec={'job':repr(job),'gpu':gpu,'architecture_status':'failed','loss_status':'failed','error':repr(e)}
            with lock: rows.append(rec)
            q.task_done()
    threads=[threading.Thread(target=loop,args=(g,),daemon=True) for g in gpu_ids]
    for t in threads: t.start()
    for t in threads: t.join()
    return pd.DataFrame(rows)

run_manifest=run_parallel_jobs([(s,f) for s in SEEDS for f in FEATURES])
display(run_manifest.sort_values(['feature','seed']))
if not (
    run_manifest.architecture_status.isin(['ok','skipped_verified']).all()
    and run_manifest.loss_status.isin(['ok','skipped_verified']).all()
):
    raise RuntimeError('One or more Pareto component jobs failed.')


## 7. Collect predictions and audit the two suites
All eight A–H architecture arms and all five loss/share controls are collected into one downstream Pareto table. A `suite` column keeps the two controlled experiments distinct.

For the loss suite, the notebook also re-checks the identical-initialization and parameter-count invariant across the four shared loss arms.


In [ ]:
raw_frames=[]; spearman_frames=[]; audit_frames=[]; diag_frames=[]

# Architecture A-H outputs.
ARCH_META = {
    'shared_base':('shared',False,False),
    'shared_dom':('shared',True,False),
    'shared_dom_pcgrad':('shared',True,True),
    'shared_dom_private':('private',True,False),
    'full_routed':('private',True,True),
    'independent_st_dom':('independent',True,False),
    'shared_capacity_dom':('capacity_shared',True,False),
    'shared_capacity_dom_pcgrad':('capacity_shared',True,True),
}
for seed in SEEDS:
    for feature in FEATURES:
        root=ARCH_RUN_DIR/f'seed_{seed}'/feature
        for arm in ARCH_ARMS:
            d=root/arm
            raw=pd.read_csv(d/'raw_predictions.csv.gz')
            raw['suite']='architecture_8arm'
            raw_frames.append(raw)

            met=pd.read_csv(d/'metrics.csv')
            met['suite']='architecture_8arm'
            spearman_frames.append(met)

            pc=pd.read_csv(d/'parameter_count.csv')
            architecture,dominance,pcgrad=ARCH_META[arm]
            pc['suite']='architecture_8arm'
            pc['architecture']=architecture
            pc['dominance_enabled']=dominance
            pc['pcgrad_enabled']=pcgrad
            pc['ranking_enabled']=True
            pc['gap_enabled']=True
            pc['arm_label']=ARCH_ARM_LABELS[arm]
            audit_frames.append(pc)

            dg=pd.read_csv(d/'training_diagnostics.csv.gz')
            dg['suite']='architecture_8arm'
            diag_frames.append(dg)

# Focal/ranking/gap suite.
for seed in SEEDS:
    for feature in FEATURES:
        root=LOSS_RUN_DIR/f'seed_{seed}'/feature
        for arm in LOSS_ARMS:
            d=root/arm
            raw=pd.read_csv(d/'raw_predictions.csv.gz')
            raw['suite']='loss_ablation'
            raw_frames.append(raw)

            met=pd.read_csv(d/'external_spearman.csv')
            met['suite']='loss_ablation'
            spearman_frames.append(met)

            aud=pd.read_csv(d/'model_audit.csv')
            aud['suite']='loss_ablation'
            audit_frames.append(aud)

            dg=pd.read_csv(d/'training_diagnostics.csv.gz')
            dg['suite']='loss_ablation'
            diag_frames.append(dg)

external_raw=pd.concat(raw_frames,ignore_index=True)
external_spearman=pd.concat(spearman_frames,ignore_index=True,sort=False)
model_audit=pd.concat(audit_frames,ignore_index=True,sort=False)
training_diag=pd.concat(diag_frames,ignore_index=True,sort=False)

# Fairness audit for the four shared loss arms.
shared_loss_arms=['S0_shared_focal','SR_shared_focal_ranking','SG_shared_focal_gap','SRG_shared_full']
loss_audit=model_audit[(model_audit.suite=='loss_ablation') & model_audit.arm.isin(shared_loss_arms)]
for (seed,feature),g in loss_audit.groupby(['seed','feature']):
    hashes=set(g.initial_param_sha256.dropna().astype(str))
    params=set(g.trainable_parameters.astype(int))
    if len(hashes)!=1:
        raise RuntimeError(f'Shared loss arms have different initializations: seed={seed} feature={feature}')
    if len(params)!=1:
        raise RuntimeError(f'Shared loss arms have different parameter counts: seed={seed} feature={feature}')
print('Shared loss-arm initialization/parameter-count audit PASS')

# Verify all eight architecture arms are present for every seed.
arch_counts=external_raw[external_raw.suite=='architecture_8arm'].groupby(['seed','feature'])['arm'].nunique()
if not (arch_counts==len(ARCH_ARMS)).all():
    raise RuntimeError('Not all eight A-H architecture arms are present.')
print('Eight-arm architecture completeness audit PASS')

external_raw.to_csv(ANALYSIS_DIR/'pareto_all_components_external_raw_predictions.csv.gz',index=False,compression='gzip')
external_spearman.to_csv(ANALYSIS_DIR/'pareto_all_components_external_spearman_by_seed.csv',index=False)
model_audit.to_csv(ANALYSIS_DIR/'pareto_all_components_model_audit.csv',index=False)
training_diag.to_csv(ANALYSIS_DIR/'pareto_all_components_training_diagnostics.csv.gz',index=False,compression='gzip')

spearman_summary=external_spearman.groupby(
    ['suite','dataset','feature','arm','arm_label'],as_index=False
).agg(
    aff_spearman_mean=('aff_spearman','mean'),
    aff_spearman_sd=('aff_spearman','std'),
    ova_spearman_mean=('ova_spearman','mean'),
    ova_spearman_sd=('ova_spearman','std'),
)
spearman_summary.to_csv(ANALYSIS_DIR/'pareto_all_components_external_spearman_summary.csv',index=False)

display(model_audit.sort_values(['suite','seed','arm']))
display(spearman_summary.sort_values(['suite','dataset','arm']))


## 8. Fixed-budget Pareto evaluation for **all arms**
**No averaging over K.** Every A–H architecture arm and every loss/share arm is evaluated independently at K = 5, 10, 15, 20, 25.

Selection is by nondominated fronts in predicted raw-logit score space with crowding-distance truncation. Evaluation uses the true continuous measurements after normalizing target desirability and inverse-OVA desirability within each dataset.


In [ ]:
def pareto_mask_max(points):
    points=np.asarray(points,float)
    mask=np.ones(len(points),bool)
    for i in range(len(points)):
        if (np.all(points>=points[i],axis=1)&np.any(points>points[i],axis=1)).any():
            mask[i]=False
    return mask


def nondominated_sort(points):
    remaining=np.arange(len(points)); fronts=[]
    while len(remaining):
        mask=pareto_mask_max(points[remaining])
        fronts.append(remaining[mask])
        remaining=remaining[~mask]
    return fronts


def crowding_distance(points):
    points=np.asarray(points,float); d=np.zeros(len(points),float)
    if len(points)<=2:
        d[:]=np.inf
        return d
    for obj in range(points.shape[1]):
        order=np.argsort(points[:,obj],kind='mergesort')
        d[order[0]]=d[order[-1]]=np.inf
        span=points[order[-1],obj]-points[order[0],obj]
        if span<=0: continue
        for r in range(1,len(points)-1):
            cur=order[r]
            if np.isfinite(d[cur]):
                d[cur]+=(points[order[r+1],obj]-points[order[r-1],obj])/span
    return d


def select_fixed_budget(points, identifiers, k):
    points=np.asarray(points,float); ids=np.asarray(identifiers).astype(str); selected=[]
    for front in nondominated_sort(points):
        if len(selected)+len(front)<=k:
            selected.extend(front.tolist())
            continue
        rem=k-len(selected)
        dist=crowding_distance(points[front])
        order=sorted(range(len(front)),key=lambda j:(-dist[j],ids[front[j]]))
        selected.extend(front[order[:rem]].tolist())
        break
    return np.asarray(selected,int)


def normalize_true_objectives(a,o):
    p=np.c_[np.asarray(a,float),-np.asarray(o,float)]
    mn=p.min(axis=0); sp=p.max(axis=0)-mn; sp[sp==0]=1.0
    return (p-mn)/sp


def hypervolume_2d_max(points):
    points=np.asarray(points,float)
    points=points[np.all(points>=0,axis=1)]
    if not len(points): return 0.0
    points=points[pareto_mask_max(points)]
    points=points[np.argsort(points[:,0])]
    total=0.0; prev=0.0
    for x,y in points:
        total+=max(0.0,x-prev)*max(0.0,y)
        prev=max(prev,x)
    return float(total)


def igd(true_front, selected):
    return float(np.sqrt(((true_front[:,None,:]-selected[None,:,:])**2).sum(axis=2)).min(axis=1).mean())

rows=[]
for (suite,seed,feature,dataset,arm,arm_label),g in external_raw.groupby(['suite','seed','feature','dataset','arm','arm_label'],sort=False):
    g=g.sort_values('row_id').reset_index(drop=True)
    pred=np.c_[g.pred_aff.to_numpy(float),-g.pred_ova.to_numpy(float)]
    true_raw=np.c_[g.true_aff.to_numpy(float),-g.true_ova.to_numpy(float)]
    true_norm=normalize_true_objectives(g.true_aff,g.true_ova)
    true_mask=pareto_mask_max(true_raw)
    ntrue=int(true_mask.sum()); prevalence=ntrue/len(g)
    for k in K_VALUES:
        if k>len(g): continue
        sel=select_fixed_budget(pred,g.sequence_id,k)
        hits=int(true_mask[sel].sum())
        precision=hits/k
        rows.append({
            'suite':suite,'seed':int(seed),'feature':feature,'dataset':dataset,'arm':arm,'arm_label':arm_label,
            'k':int(k),'n_candidates':len(g),'n_true_pareto':ntrue,'hits':hits,
            'recall_at_k':hits/max(ntrue,1),
            'precision_at_k':precision,
            'enrichment_at_k':precision/prevalence if prevalence>0 else np.nan,
            'hypervolume_true_selected':hypervolume_2d_max(true_norm[sel]),
            'igd_true_front_to_selected':igd(true_norm[true_mask],true_norm[sel]),
        })

pareto_by_run=pd.DataFrame(rows)
pareto_by_run.to_csv(ANALYSIS_DIR/'pareto_all_components_fixed_budget_by_run.csv',index=False)
pareto_summary=pareto_by_run.groupby(['suite','dataset','feature','arm','arm_label','k'],as_index=False).agg(
    n_runs=('seed','size'),
    n_true_pareto=('n_true_pareto','first'),
    hits_mean=('hits','mean'),hits_sd=('hits','std'),
    recall_mean=('recall_at_k','mean'),recall_sd=('recall_at_k','std'),
    precision_mean=('precision_at_k','mean'),precision_sd=('precision_at_k','std'),
    enrichment_mean=('enrichment_at_k','mean'),enrichment_sd=('enrichment_at_k','std'),
    hypervolume_mean=('hypervolume_true_selected','mean'),hypervolume_sd=('hypervolume_true_selected','std'),
    igd_mean=('igd_true_front_to_selected','mean'),igd_sd=('igd_true_front_to_selected','std'),
)
pareto_summary.to_csv(ANALYSIS_DIR/'pareto_all_components_fixed_budget_summary.csv',index=False)

for dataset in ['ISO','IgG-primary42','IgG-all96']:
    print('\n',dataset)
    display(pareto_summary[pareto_summary.dataset==dataset].sort_values(['suite','k','arm']))

## 9. Component-attribution contrasts at every K
Positive `oriented_mean_advantage` means the **left-hand arm is better**. For IGD the sign is reversed internally because lower IGD is better.

### Loss-suite contrasts
- ranking alone: `SR - S0`
- gap alone: `SG - S0`
- combined ranking+gap: `SRG - S0`
- incremental gap given ranking: `SRG - SR`
- incremental ranking given gap: `SRG - SG`
- parameter sharing under matched full objective: `SRG - IRG`

### Architecture-suite contrasts
The notebook reports:
- all **7 Full Routed-MOLM vs control** comparisons, matching the prior A–H ablation logic;
- targeted effects for dominance, PCGrad, private paths, capacity matching, and sharing;
- **all 28 pairwise comparisons among the eight A–H arms** at every K and every Pareto metric.

With five optimization seeds, exact sign-flip p-values are coarse; they are reported as reproducibility/statistical diagnostics and are not treated as the sole basis for interpretation.


In [ ]:
def exact_sign_flip_pvalue(diffs):
    d=np.asarray(diffs,float)
    d=d[np.isfinite(d)]
    if len(d)==0: return np.nan
    observed=abs(d.mean())
    vals=[]
    for mask in range(1<<len(d)):
        signs=np.array([1.0 if (mask>>i)&1 else -1.0 for i in range(len(d))])
        vals.append(abs((d*signs).mean()))
    vals=np.asarray(vals)
    return float((vals>=observed-1e-15).mean())

def holm_adjust(pvals):
    p=np.asarray(pvals,float); out=np.full(len(p),np.nan)
    valid=np.flatnonzero(np.isfinite(p))
    if not len(valid): return out
    order=valid[np.argsort(p[valid])]
    running=0.0; m=len(order)
    for rank,idx in enumerate(order):
        adjusted=(m-rank)*p[idx]
        running=max(running,adjusted)
        out[idx]=min(1.0,running)
    return out

METRICS = {
    'recall_at_k':'higher',
    'precision_at_k':'higher',
    'enrichment_at_k':'higher',
    'hypervolume_true_selected':'higher',
    'igd_true_front_to_selected':'lower',
}

def build_contrasts(source_df, suite, contrast_specs):
    rows=[]
    sdf=source_df[source_df.suite==suite]
    for (dataset,feature,k),g in sdf.groupby(['dataset','feature','k']):
        for contrast,left,right in contrast_specs:
            for metric,direction in METRICS.items():
                piv=g.pivot_table(index='seed',columns='arm',values=metric,aggfunc='first').dropna(subset=[left,right])
                raw=(piv[left]-piv[right]).to_numpy(float)
                oriented=raw if direction=='higher' else -raw
                rows.append({
                    'suite':suite,'dataset':dataset,'feature':feature,'k':int(k),
                    'contrast':contrast,'left_arm':left,'right_arm':right,
                    'metric':metric,'direction':direction,'n_seeds':len(oriented),
                    'raw_mean_left_minus_right':float(raw.mean()),
                    'oriented_mean_advantage':float(oriented.mean()),
                    'oriented_sd':float(oriented.std(ddof=1)) if len(oriented)>1 else np.nan,
                    'wins_left':int((oriented>0).sum()),
                    'ties':int(np.isclose(oriented,0).sum()),
                    'losses_left':int((oriented<0).sum()),
                    'exact_sign_flip_p':exact_sign_flip_pvalue(oriented),
                })
    return pd.DataFrame(rows)

LOSS_CONTRASTS = [
    ('ranking_alone','SR_shared_focal_ranking','S0_shared_focal'),
    ('gap_alone','SG_shared_focal_gap','S0_shared_focal'),
    ('combined_aux','SRG_shared_full','S0_shared_focal'),
    ('gap_given_ranking','SRG_shared_full','SR_shared_focal_ranking'),
    ('ranking_given_gap','SRG_shared_full','SG_shared_focal_gap'),
    ('sharing_full_objective','SRG_shared_full','IRG_independent_full'),
]

ARCH_TARGETED_CONTRASTS = [
    ('routed_vs_shared_base','full_routed','shared_base'),
    ('routed_vs_shared_dom','full_routed','shared_dom'),
    ('routed_vs_shared_dom_pcgrad','full_routed','shared_dom_pcgrad'),
    ('routed_vs_shared_dom_private','full_routed','shared_dom_private'),
    ('routed_vs_independent_matched','full_routed','independent_st_dom'),
    ('routed_vs_capacity_matched','full_routed','shared_capacity_dom'),
    ('routed_vs_capacity_pcgrad','full_routed','shared_capacity_dom_pcgrad'),
    ('sharing_matched_joint_loss','shared_dom','independent_st_dom'),
    ('dominance_effect','shared_dom','shared_base'),
    ('pcgrad_effect','shared_dom_pcgrad','shared_dom'),
    ('private_path_effect','shared_dom_private','shared_dom'),
    ('pcgrad_on_private_routing','full_routed','shared_dom_private'),
    ('capacity_effect','shared_capacity_dom','shared_dom'),
    ('pcgrad_capacity_effect','shared_capacity_dom_pcgrad','shared_capacity_dom'),
]

loss_contrasts=build_contrasts(pareto_by_run,'loss_ablation',LOSS_CONTRASTS)
arch_targeted=build_contrasts(pareto_by_run,'architecture_8arm',ARCH_TARGETED_CONTRASTS)

# Holm correction: loss = six predeclared contrasts; routed-vs-control = seven; all targeted = reported separately.
out=[]
for _,g in loss_contrasts.groupby(['dataset','feature','k','metric'],sort=False):
    g=g.copy(); g['exact_sign_flip_p_holm_family']=holm_adjust(g.exact_sign_flip_p.to_numpy(float)); out.append(g)
loss_contrasts=pd.concat(out,ignore_index=True)

out=[]
for _,g in arch_targeted.groupby(['dataset','feature','k','metric'],sort=False):
    g=g.copy()
    routed_mask=g.contrast.str.startswith('routed_vs_')
    g['exact_sign_flip_p_holm_routed7']=np.nan
    if routed_mask.any():
        g.loc[routed_mask,'exact_sign_flip_p_holm_routed7']=holm_adjust(g.loc[routed_mask,'exact_sign_flip_p'].to_numpy(float))
    out.append(g)
arch_targeted=pd.concat(out,ignore_index=True)

# All 28 pairwise A-H comparisons.
from itertools import combinations
ARCH_ALL_PAIRS=[(f'{a}_vs_{b}',a,b) for a,b in combinations(ARCH_ARMS,2)]
arch_pairwise=build_contrasts(pareto_by_run,'architecture_8arm',ARCH_ALL_PAIRS)
out=[]
for _,g in arch_pairwise.groupby(['dataset','feature','k','metric'],sort=False):
    g=g.copy(); g['exact_sign_flip_p_holm_28']=holm_adjust(g.exact_sign_flip_p.to_numpy(float)); out.append(g)
arch_pairwise=pd.concat(out,ignore_index=True)

loss_contrasts.to_csv(ANALYSIS_DIR/'pareto_loss_component_contrasts_all_k.csv',index=False)
arch_targeted.to_csv(ANALYSIS_DIR/'pareto_architecture_targeted_contrasts_all_k.csv',index=False)
arch_pairwise.to_csv(ANALYSIS_DIR/'pareto_architecture_all28_pairwise_all_k.csv',index=False)

# Compact analysis-facing union.
component_contrasts=pd.concat([
    loss_contrasts,
    arch_targeted[arch_targeted.contrast.isin([
        'routed_vs_shared_base','routed_vs_independent_matched',
        'sharing_matched_joint_loss','dominance_effect','pcgrad_effect',
        'private_path_effect','capacity_effect'
    ])],
],ignore_index=True,sort=False)
component_contrasts.to_csv(ANALYSIS_DIR/'pareto_component_contrasts_all_k.csv',index=False)

for dataset in ['ISO','IgG-primary42','IgG-all96']:
    print('\nLOSS CONTRASTS:',dataset)
    display(loss_contrasts[loss_contrasts.dataset==dataset].sort_values(['k','metric','contrast']))
    print('\nA-H TARGETED CONTRASTS:',dataset)
    display(arch_targeted[arch_targeted.dataset==dataset].sort_values(['k','metric','contrast']))


## 10. Summary tables — every K explicit


In [ ]:
def fmt(x):
    return 'NA' if pd.isna(x) else f'{x:.4f}'

summary_rows=[]
for _,r in pareto_summary.sort_values(['suite','dataset','feature','k','arm']).iterrows():
    summary_rows.append({
        'suite':r.suite,'dataset':r.dataset,'feature':r.feature,'k':int(r.k),
        'arm':r.arm,'arm_label':r.arm_label,
        'Recall/Precision/Enrichment/HV/IGD':'/'.join([
            fmt(r.recall_mean),fmt(r.precision_mean),fmt(r.enrichment_mean),
            fmt(r.hypervolume_mean),fmt(r.igd_mean)
        ])
    })
summary_table=pd.DataFrame(summary_rows)
summary_table.to_csv(ANALYSIS_DIR/'pareto_all_components_summary_all_k.csv',index=False)

architecture_summary_table=summary_table[summary_table.suite=='architecture_8arm'].copy()
loss_summary_table=summary_table[summary_table.suite=='loss_ablation'].copy()
architecture_summary_table.to_csv(ANALYSIS_DIR/'pareto_8arm_architecture_summary_all_k.csv',index=False)
loss_summary_table.to_csv(ANALYSIS_DIR/'pareto_loss_ablation_summary_all_k.csv',index=False)

print('Eight-arm architecture table:')
display(architecture_summary_table)
print('Loss/share table:')
display(loss_summary_table)


## 11. Final reproducibility bundle
The ZIP contains:

- raw external predictions for **all eight A–H architecture arms** and all loss/share controls;
- per-seed and summary fixed-budget Pareto metrics at K=5,10,15,20,25;
- all 28 A–H pairwise comparisons plus targeted Routed/component contrasts;
- ranking/gap/share contrasts;
- parameter-count and initialization audits;
- training diagnostics;
- both embedded workers and the exact A–H helper module;
- the experiment lock and hashes.


In [ ]:
# Copy lock + both workers + A-H helper into the analysis directory.
shutil.copy2(LOCK_PATH, ANALYSIS_DIR/LOCK_PATH.name)
shutil.copy2(WORK_ROOT/'PARETO_COMPONENT_ABLATION_LOCK.sha256', ANALYSIS_DIR/'PARETO_COMPONENT_ABLATION_LOCK.sha256')
shutil.copy2(COMMON_PATH, ANALYSIS_DIR/COMMON_PATH.name)
shutil.copy2(ARCH_WORKER_PATH, ANALYSIS_DIR/ARCH_WORKER_PATH.name)
shutil.copy2(LOSS_WORKER_PATH, ANALYSIS_DIR/LOSS_WORKER_PATH.name)

manifest_rows=[]
for p in sorted(ANALYSIS_DIR.iterdir()):
    if p.is_file():
        manifest_rows.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)})
output_manifest=pd.DataFrame(manifest_rows)
output_manifest.to_csv(ANALYSIS_DIR/'PARETO_ALL_COMPONENTS_OUTPUT_MANIFEST.csv',index=False)

ZIP_PATH=Path('/kaggle/working/MOLM_Pareto_Component_Loss_Ablation_Results.zip')
if ZIP_PATH.exists(): ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(ANALYSIS_DIR.iterdir()):
        if p.is_file():
            z.write(p,arcname=p.name)

print('Final bundle:', ZIP_PATH)
print('Size MB:', ZIP_PATH.stat().st_size/1024**2)
display(output_manifest)


## Interpretation guardrails
Use this notebook to answer the component question **only after seeing the downstream Pareto results**:

- The **eight-arm A–H suite** is the downstream counterpart of the completed mutation-site architecture/component ablation. It includes **Full Routed-MOLM** and all seven prior controls.
- The **loss suite** isolates focal, ranking, and gap effects. Do not infer a Pareto effect from MCC alone.
- `shared_base` (architecture suite) and `SRG_shared_full` (loss suite) are analyzed within their own controlled suites; do **not** use cross-suite differences as a causal contrast because the workers include different diagnostics/training control logic.
- If ranking/gap change HV, IGD, enrichment, precision, or recall at fixed K despite little MCC change, that supports a **downstream prioritization role** for those objectives.
- If shared and independent matched-objective models differ in Pareto metrics, that quantifies the effect of **sharing on candidate prioritization**, not merely classification.
- If Routed-MOLM differs from Shared Base or matched independent controls in Pareto space, report the direction **at each K** rather than making a global claim.
- If all arms remain similar, conclude that the simpler formulation is sufficient; do not manufacture an advantage.
- Keep every `K = 5, 10, 15, 20, 25` explicit. **Do not average over K.**
